# When Can We Trust an Internal Direction? | Phi-3 final server experiments

**One reproducible Jupyter notebook, no server Jupyter, Drive mounts, uploads, or private archive dependencies.**

**Main experiments:** (1) held-out formulation transfer on IFEval-Simple and FollowBench-derived settings; (2) matched comparisons of B0/B2/MT0/MT2 with within-type, text-only, type-only and grouped uncertainty controls; (3) observed compliance changes on matched formulation pairs. **CCT extension:** a controlled 2 × 2 paired requirement/wording diagnostic, rebuilt from the **single local CCT benchmark ZIP** supplied by the user, using a new independently fitted set of the same four readout *types*.

**Only non-public input:** `cct_benchmark.zip` in this notebook's working directory (or specify an absolute `ULM_CCT_ZIP` environment variable). It is **not** a precomputed responses archive; no old local disk datasets, trained probes, or state files are needed. All public data, the official instruction verifier, and the pinned public Phi-3 checkpoint are downloaded automatically. A network connection is required, but **no sign-in prompt, user upload or manual file selection is used**.

**GPU:** single NVIDIA Ampere-class BF16 GPU (A100/H100 or equivalent), using sequential frozen inference. Run the notebook top-to-bottom on a Jupyter kernel on the server. Set `ULM_WORKDIR` if the default output folder is unsuitable. Existing files with matching fingerprints are reused; incompatible cached results cause an error rather than silent mixing.

**CCT comparability warning:** the earlier CCT publication-prototype evaluated *historically frozen source-study probes*. Those source-study archives are not included in the publicly available CCT benchmark. To satisfy the one-ZIP requirement, this notebook instead independently fits B0/B2/MT0/MT2 using CCT's task-family-disjoint training folds on O (canonical) and R1 (reformulation). It then evaluates out-of-fold O/R1 and withheld C0/C1, without training on counterfactual outcomes. It is a **new, reproducible CCT variant**, not an exact replay of the archived CCT probe parameters or old numeric effect sizes. The historical three-setting source implementations are preserved and can be compared to their original runs.

**Important definitions:** the FollowBench-derived setting use **additional verified constraints**, not the benchmarks' native joint task scores. Automated verifier success does not certify full task completion or semantic compatibility. The notebook does not present run results until they have been generated on the server.
**Packaged execution:** Run from `notebooks/` after unpacking. The bundled CCT asset is detected at `../data/cct_benchmark.zip`. For other working directories set `ULM_CCT_ZIP` explicitly. Model revision pins and disk caches are retained across restarts. Never use the same `ULM_WORKDIR` for different checkpoints.


## 0. Server prerequisites and public download preflight

Configure only two local paths. Dependencies install in the current Jupyter kernel. An ordinary `pip install` and network connection to GitHub/Hugging Face are sufficient. The CCT benchmark ZIP is the **only** user-provided input. Source revisions and study protocols are checked before inference.

In [ ]:


import os,sys,hashlib,json
from pathlib import Path
_MODEL_LABEL='microsoft/Phi-3-mini-128k-instruct'
_CODE_RELEASE_SHA='f43b33f51761fd329dec63c0b44c8c4223951655342e286898ef016bf1fc6991'
_REPO_ROOT=Path('..') if Path('../data/cct_benchmark.zip').exists() else Path('.')
_WORKDIR_DEFAULT=_REPO_ROOT/'results'/('ulm_'+'phi3'+'_server_results')
_ULM_OUT=Path(os.environ.get('ULM_WORKDIR',str(_WORKDIR_DEFAULT))).expanduser().resolve()
_ULM_DATA=Path(os.environ.get('ULM_CCT_ZIP',str(_REPO_ROOT/'data/cct_benchmark.zip'))).expanduser().resolve()
_RUN_MARKER=_ULM_OUT/'COMPLETED_RUN.json'
def _fingerprint_file(path):
    h=hashlib.sha256()
    with path.open('rb') as reader:
        for part in iter(lambda:reader.read(1024*1024),b''):h.update(part)
    return h.hexdigest()
def _replay_valid():
    if not _RUN_MARKER.is_file() or not _ULM_DATA.is_file():return False,'No complete marker or CCT input'
    try:
        m=json.loads(_RUN_MARKER.read_text())
        if m['version']!='verified-v2' or m['model']!=_MODEL_LABEL or m['notebook_sha256']!=_CODE_RELEASE_SHA:return False,'Protocol changed'
        if m['cct_sha256']!=_fingerprint_file(_ULM_DATA):return False,'CCT changed'
        entries=m['files']
        if not entries:return False,'Missing output inventory'
        for relative,rec in entries.items():
            p=_ULM_OUT/relative
            if not p.is_file() or p.stat().st_size!=rec['size']:return False,'Missing or truncated '+relative
            if rec.get('sha256') and _fingerprint_file(p)!=rec['sha256']:return False,'Changed '+relative
        return True,'Validated '+str(len(entries))+' saved files'
    except Exception as e:return False,'Corrupt/incompatible marker: '+str(e)
REPLAY_COMPLETE,_REPLAY_REASON=_replay_valid()
print('MODEL:',_MODEL_LABEL,'| completed-run replay:',REPLAY_COMPLETE,'|',_REPLAY_REASON)
if REPLAY_COMPLETE:
    print('All expected study artifacts found. Skipping package installation, downloads, inference, training, and analysis.')
    print('Results:',_ULM_OUT/'final_tables')
    print('CCT:',_ULM_OUT/'CCT_independent_refit')


In [ ]:
if not REPLAY_COMPLETE:
    
    import importlib.metadata as _im, sys as _sys, subprocess as _sp
    _needed={'transformers':'4.53.3','accelerate':None,'huggingface-hub':None,
     'sentencepiece':None,'scikit-learn':None,'pandas':None,'scipy':None,'tqdm':None,
     'nltk':None,'immutabledict':None,'langdetect':None,'absl-py':None,'requests':None,
     'matplotlib':None,'emoji':None}
    _missing=[]
    for _pkg,_exact in _needed.items():
        try:
            _got=_im.version(_pkg)
            if _exact is not None and _got!=_exact:_missing.append((_pkg,_got,_exact))
        except _im.PackageNotFoundError:_missing.append((_pkg,'missing',_exact))
    if _missing:
        print('Installing missing/incompatible dependencies:',_missing)
        import shlex as _shlex
        _sp.check_call([_sys.executable,'-m','pip']+_shlex.split("-q install 'transformers==4.53.3' 'accelerate>=1.2,<2' 'huggingface-hub>=0.30,<1' 'sentencepiece>=0.2' 'scikit-learn>=1.4,<1.8' 'pandas>=2,<3' 'scipy>=1.12,<2' 'tqdm>=4.66' 'nltk>=3.8,<4' 'immutabledict>=4' 'langdetect>=1.0.9' 'absl-py>=2' 'requests>=2.31' 'matplotlib>=3.7' 'emoji>=2.10,<3'"))
    else:
        print('Python dependencies already satisfied; no installation or network required.')

In [ ]:
if not REPLAY_COMPLETE:
    from pathlib import Path
    import os, zipfile, json, hashlib
    
    _REPOSITORY_ROOT = (Path('..') if Path('../data/cct_benchmark.zip').exists()
         else Path('.') if Path('data/cct_benchmark.zip').exists() else Path('..'))
    _DEFAULT_WORKDIR = _REPOSITORY_ROOT / 'results' / 'ulm_phi3_server_results'
    ULM_WORKDIR = Path(os.environ.get('ULM_WORKDIR',str(_DEFAULT_WORKDIR))).expanduser().resolve()
    ULM_WORKDIR.mkdir(parents=True,exist_ok=True)
    
    _NOTEBOOK_CCT_CANDIDATES=[Path('data/cct_benchmark.zip'),
     Path('../data/cct_benchmark.zip'),Path('./cct_benchmark.zip')]
    _CCT_AUTO=next((x for x in _NOTEBOOK_CCT_CANDIDATES if x.is_file()),_NOTEBOOK_CCT_CANDIDATES[0])
    CCT_ZIP = Path(os.environ.get('ULM_CCT_ZIP',str(_CCT_AUTO))).expanduser().resolve()
    assert CCT_ZIP.is_file(), ("Place cct_benchmark.zip beside this notebook or set ULM_CCT_ZIP="
                                "/absolute/path/to/cct_benchmark.zip. No other inputs are required.")
    with zipfile.ZipFile(CCT_ZIP) as z:
        assert z.testzip() is None, 'Corrupted CCT benchmark ZIP'
        expected={'benchmark_protocol.json','SHA256SUMS.json','unified/CCT_counterfactual_prompts.jsonl'}
        assert expected.issubset(z.namelist()), ("Wrong input: expected the CCT benchmark data ZIP, not the"
                                               " CCT final-results ZIP",sorted(expected-set(z.namelist())))
        declared=json.loads(z.read('SHA256SUMS.json'))
        assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in declared.items()), 'CCT ZIP member hash mismatch'
    _PINNED_CHECKER_CANDIDATES=[Path('data/official_checker/instruction_following_eval'),
        Path('../data/official_checker/instruction_following_eval')]
    _LOCAL_CHECKER=next((d for d in _PINNED_CHECKER_CANDIDATES if (d/'instructions.py').is_file()), None)
    if _LOCAL_CHECKER is not None and not os.environ.get('ULM_CHECKER_SOURCE_DIR'):
        os.environ['ULM_CHECKER_SOURCE_DIR']=str(_LOCAL_CHECKER.resolve())
    print('Server output folder:',ULM_WORKDIR)
    print('CCT input:',CCT_ZIP,'| SHA256:',hashlib.sha256(CCT_ZIP.read_bytes()).hexdigest())
    print('CCT manifest member hash checks: PASS')

# Part I. IFEval-Simple: source reproduction and matched reformulations

**From scratch:** 102 independent base tasks, three prespecified verifiable instruction types, and six formulations (`original`, `P1`–`P3`, held-out `R1`–`R2`). Use the original nested group splits, 1,024-token generation cap, canonical layer 14, frozen four-probe comparison, grouped uncertainty, and matched flip tests. The pinned model weights are loaded for generation only and then released before probing.

In [ ]:
if not REPLAY_COMPLETE:
    
    MODEL = 'phi'           
    RUN_TAG = 'final_phi3_simple_v1'  
    RUN_TEXT_CONTROL = True
    RUN_OTHER_LAYERS = False    
    N_BOOT = 2000
    SEED = 20261017

    
    MODEL_CONFIGS = {
        'phi': {'model_id':'microsoft/Phi-3-mini-128k-instruct',
                'layers':(14,26,32),'expected_hidden':3072,'expected_layers':32,
                'family':'phi3'},
    }
    assert MODEL in MODEL_CONFIGS, f'Unknown MODEL={MODEL!r}'
    CFG = MODEL_CONFIGS[MODEL]
    MODEL_ID = CFG['model_id']
    LAYERS = CFG['layers']
    PRIMARY_LAYER = 14
    GENERATION_BUDGET = 1024             
    MAX_INPUT_TOKENS = 1024
    FOLDS = 5
    Cs = (0.03,0.1,0.3)
    DELTA_PENALTIES = (4.0,16.0)
    ORIG_MARGIN = 0.02
    METHODS = ('B0','B2','MT0','MT2')
    TYPES = ['keywords:forbidden_words','keywords:existence','keywords:frequency',
             'detectable_content:number_placeholders','startend:end_checker']
    
    PRIMARY_TYPES = ('keywords:forbidden_words','detectable_content:number_placeholders','startend:end_checker')
    R_TEMPLATES = ('R1','R2')
    print('Chosen model:', MODEL, '| Hugging Face repo:', MODEL_ID)

In [ ]:
if not REPLAY_COMPLETE:
    
    import os, sys, gc, math, json, time, itertools, hashlib, warnings, zipfile, shutil, random, importlib
    from pathlib import Path
    from collections import Counter
    from datetime import datetime, timezone
    import numpy as np, pandas as pd, torch, requests, nltk, transformers
    from tqdm.auto import tqdm
    from IPython.display import display
    from sklearn.model_selection import KFold, train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
    from scipy.special import expit
    from scipy.stats import binomtest
    from huggingface_hub import HfApi

    assert transformers.__version__ == '4.53.3', ("Wrong loaded Transformers version: ",transformers.__version__," Restart runtime and rerun first 2 cells.")
    assert torch.cuda.is_available(), 'CUDA GPU required for the frozen BF16 inference protocol.'
    assert torch.cuda.is_bf16_supported(), 'BF16 GPU required to reproduce our standard protocol; choose A100 or better.'
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    DTYPE=torch.bfloat16
    GPU_NAME=torch.cuda.get_device_name(0)
    print('GPU',GPU_NAME,'| torch',torch.__version__,'| transformers',transformers.__version__)

    
    PHI_LEGACY_ROOT=ULM_WORKDIR/'IFEval-Simple'
    DRIVE_BASE=PHI_LEGACY_ROOT/'source_replication'
    MODEL_ROOT=DRIVE_BASE/MODEL
    DATA_DIR=PHI_LEGACY_ROOT/'data'
    for p in (MODEL_ROOT,DATA_DIR):p.mkdir(parents=True,exist_ok=True)
    MODEL_REV='f3c06aed622e14ca0abf5115094e4fc9a9948f36'
    REV_RECORD=MODEL_ROOT/'selected_revision.json'
    if REV_RECORD.exists():
        saved=json.loads(REV_RECORD.read_text())
        assert saved=={'model_id':MODEL_ID,'revision':MODEL_REV},'Model revision changed in this workspace'
    else:
        REV_RECORD.write_text(json.dumps({'model_id':MODEL_ID,'revision':MODEL_REV},indent=2))
    print('Pinned revision:',MODEL_REV)

    def sha(buf):return hashlib.sha256(buf).hexdigest()
    def canonical(obj):return json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False)
    def shafile(p):
        h=hashlib.sha256()
        with open(p,'rb') as f:
            for buf in iter(lambda:f.read(1024*1024),b''):h.update(buf)
        return h.hexdigest()
    def atomic_json(p,obj):
        p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp')
        tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=str),encoding='utf-8')
        os.replace(tmp,p)
    def atomic_npz(p,**arrays):
        p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp')
        with open(tmp,'wb') as f:np.savez_compressed(f,**arrays)
        os.replace(tmp,p)
    def freeze_json(p,obj):
        p=Path(p)
        if p.exists(): assert json.loads(p.read_text(encoding='utf-8'))==json.loads(canonical(obj)), f'Frozen protocol mismatch at {p}. Change RUN_TAG to start a new run.'
        else:atomic_json(p,obj)
    def safe_auc(y,p):
        y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
        return float(roc_auc_score(y,p)) if len(np.unique(y))==2 else float('nan')
    auc=safe_auc  
    for pkg in ('punkt','punkt_tab'):assert nltk.download(pkg,quiet=True),pkg

### 1.1 Source prompts and verifiers

Dataset and official IFEval checker files are downloaded automatically from public repositories, hashed, and validated. Rewrite templates and split assignments are frozen before outcomes are measured.

In [ ]:
if not REPLAY_COMPLETE:
    
    
    DATA_URL = 'https://raw.githubusercontent.com/apple-aiml-research/ml-internal-llms-instruction-following/main/data/ifeval_simple.jsonl'
    DATA_FILE=DATA_DIR/'ifeval_simple.jsonl'
    CHECKER_FILES=('instructions.py','instructions_util.py','instructions_registry.py')
    CHECKER_DIR=DATA_DIR/'instruction_following_eval'
    CHECKER_DIR.mkdir(parents=True,exist_ok=True)
    (CHECKER_DIR/'__init__.py').touch(exist_ok=True)
    CHECKER_BASE='https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/'

    def fetch(url,p):
        p=Path(p)
        if p.is_file() and p.stat().st_size>0:return 'existing'
        local_checker=Path(os.environ.get('ULM_CHECKER_SOURCE_DIR',''))
        if url.startswith(CHECKER_BASE) and os.environ.get('ULM_CHECKER_SOURCE_DIR'):
            from hashlib import sha256
            pinned=local_checker/p.name
            expected={'instructions.py':'60e086f5342a03ce8e18b64bbcccf86308f523c08aa826707a562150a52f3edf',
              'instructions_util.py':'a73797261eee5bf447e279d82a2b700b1bdd3cb1193412dbab1270a85832bc6b',
              'instructions_registry.py':'ec92d72c264f6d906978613085db262356174300370a3fffe6fefd5969ce9cfc'}
            if pinned.is_file() and sha256(pinned.read_bytes()).hexdigest()==expected.get(p.name):
                p.parent.mkdir(parents=True,exist_ok=True)
                p.write_bytes(pinned.read_bytes())
                return 'bundled pinned official checker'

        r=requests.get(url,timeout=120);r.raise_for_status()
        p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp')
        tmp.write_bytes(r.content)
        os.replace(tmp,p)
        return 'downloaded'
    
    
    
    import io
    SIMPLE_REPO='apple-aiml-research/ml-internal-llms-instruction-following'
    def _parse_simple_dataset(blob):
        try:
            obj=json.loads(blob.decode('utf8'))
            if isinstance(obj,dict):
                candidates=[obj[k] for k in ['examples','data','records','items'] if k in obj]
                obj=next((a for a in candidates if isinstance(a,list)),obj)
            if isinstance(obj,list):return obj
        except (ValueError,UnicodeDecodeError):pass
        try:return [json.loads(s) for s in blob.decode('utf8').splitlines() if s.strip()]
        except (ValueError,UnicodeDecodeError):return None

    def _valid_simple(rows):
        if not isinstance(rows,list) or len(rows)!=510:return False
        return all(isinstance(x,dict) and 'prompt' in x and 'instruction_id_list' in x
                   and 'kwargs' in x for x in rows)

    if DATA_FILE.exists():
        probe=_parse_simple_dataset(DATA_FILE.read_bytes())
        assert _valid_simple(probe),f'Cached IFEval-Simple is not the 510-record source: {DATA_FILE}'
    else:
        known=[DATA_URL,
          f'https://raw.githubusercontent.com/{SIMPLE_REPO}/main/data/ifeval_simple.json',
          f'https://raw.githubusercontent.com/{SIMPLE_REPO}/main/data/IFEval_simple.json',
          f'https://raw.githubusercontent.com/{SIMPLE_REPO}/main/data/IFEval-Simple.json',
          f'https://raw.githubusercontent.com/{SIMPLE_REPO}/main/data/IFEval_Simple.json']
        discovered=[]
        try:
            api=requests.get(f'https://api.github.com/repos/{SIMPLE_REPO}/git/trees/main?recursive=1',timeout=90)
            api.raise_for_status()
            entries=api.json()['tree']
            files=[x['path'] for x in entries if x.get('type')=='blob' and x['path'].startswith('data/')
                 and x['path'].lower().endswith(('.json','.jsonl'))]
            ranked=sorted(files,key=lambda x:(0 if 'simple' in x.lower() else 1, x))
            discovered=[f'https://raw.githubusercontent.com/{SIMPLE_REPO}/main/{path}' for path in ranked]
        except (requests.RequestException,KeyError,ValueError) as e:
            print('GitHub tree discovery unavailable; trying known official paths:',type(e).__name__)
        candidates=list(dict.fromkeys(known+discovered))
        found=None
        for url in candidates:
            try:
                reply=requests.get(url,timeout=120)
                if reply.status_code!=200:continue
                parsed=_parse_simple_dataset(reply.content)
                if _valid_simple(parsed):
                    found=(url,parsed)
                    break
            except requests.RequestException:continue
        assert found is not None, ('Unable to discover the official 510-record IFEval-Simple dataset. '
             'Check public GitHub network access; no alternate private file is assumed.')
        DATA_URL,probe=found
        
        norm=''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in probe)
        tmp=DATA_FILE.with_suffix('.jsonl.tmp');tmp.write_text(norm,encoding='utf8');os.replace(tmp,DATA_FILE)
        (DATA_DIR/'simple_public_source_url.txt').write_text(DATA_URL+'\n',encoding='utf8')
        print('Official dataset discovered:',DATA_URL)
    print('IFEval-Simple records validated:',len(probe),'| local copy:',DATA_FILE)
    for name in CHECKER_FILES:print('Verifier',name,fetch(CHECKER_BASE+name,CHECKER_DIR/name))
    raw=[json.loads(s) for s in DATA_FILE.read_text(encoding='utf-8').splitlines() if s.strip()]
    assert len(raw)==510, f'Expected the original 510-row IFEval-Simple source: got {len(raw)}'
    assert all(raw[i*102+j]['instruction_id_list']==[TYPES[i]] for i in range(5) for j in range(102))
    base_tasks=[raw[k]['prompt'].split('\nDo not include the following keywords:')[0].strip() for k in range(102)]
    assert len(set(base_tasks))==102
    assert all(raw[t*102+k]['prompt'].startswith(base_tasks[k]) for t in range(5) for k in range(102))
    
    if str(DATA_DIR) not in sys.path:sys.path.insert(0,str(DATA_DIR))
    from instruction_following_eval import instructions_registry
    assert set(TYPES).issubset(instructions_registry.INSTRUCTION_DICT)
    SOURCE={'dataset_sha256':shafile(DATA_FILE),
            'checker_sha256':{f:shafile(CHECKER_DIR/f) for f in CHECKER_FILES},
            'original_repo_url':DATA_URL,'model_id':MODEL_ID,'revision':MODEL_REV}
    TYPE_TO_ID={t:i for i,t in enumerate(TYPES)}
    N_TYPES=len(TYPES)

    def kwargs_one(x):
        if isinstance(x,dict):x=[x]
        assert isinstance(x,list) and len(x)==1 and isinstance(x[0],dict),x
        return x[0]

    def strict_score(record,response):
        if not isinstance(response,str) or not response.strip():return 0
        iid=record['instruction_type']
        checker=instructions_registry.INSTRUCTION_DICT[iid](iid)
        checker.build_description(**kwargs_one(record['kwargs']))
        return int(bool(checker.check_following(response)))

    def score(typ,kw,response):
        return strict_score({'instruction_type':typ,'kwargs':kw},response)
    print('Verified source: 510 rows / 102 groups / 5 official types; SHA computed at runtime')

In [ ]:
if not REPLAY_COMPLETE:
    from collections import Counter
    def phrase(instruction_id, kw, variant):
        assert variant in ('P1','P2','P3')
        q=lambda x:json.dumps(str(x),ensure_ascii=False)
        if instruction_id=='keywords:forbidden_words':
            assert set(kw)=={'forbidden_words'} and isinstance(kw['forbidden_words'],list)
            words=', '.join(q(z) for z in kw['forbidden_words'])
            templates=[f'Never include any of these keywords in the answer: {words}.',
                       f'Your response must not contain any keyword from this list: {words}.',
                       f'Constraint on your answer: avoid every one of these keywords: {words}.']
        elif instruction_id=='keywords:existence':
            assert set(kw)=={'keywords'} and isinstance(kw['keywords'],list)
            words=', '.join(q(z) for z in kw['keywords'])
            templates=[f'Include all the following keywords in your answer: {words}.',
                       f'Your response must contain each keyword in this list: {words}.',
                       f'Required keywords (include every one): {words}.']
        elif instruction_id=='keywords:frequency':
            assert set(kw)=={'relation','keyword','frequency'}
            rel=kw['relation']; n=int(kw['frequency']); w=q(kw['keyword'])
            assert rel in ('at least','less than') and n>=0
            if rel=='at least':
                templates=[f'Use the word {w} at least {n} times in your answer.',
                           f'Your response must contain {w} no fewer than {n} times.',
                           f'Keyword-count requirement: {w} must occur {n} times or more.']
            else:
                templates=[f'Use the word {w} fewer than {n} times in your answer.',
                           f'Your response must contain {w} less than {n} times.',
                           f'Keyword-count requirement: {w} must occur strictly fewer than {n} times.']
        elif instruction_id=='detectable_content:number_placeholders':
            assert set(kw)=={'num_placeholders'}
            n=int(kw['num_placeholders']); assert n>=0
            templates=[f'Include at least {n} placeholders enclosed in square brackets, such as [name].',
                       f'Your answer must contain no fewer than {n} square-bracket placeholders (for example, [name]).',
                       f'Placeholder requirement: include {n} or more placeholders of the form [name].']
        elif instruction_id=='startend:end_checker':
            assert set(kw)=={'end_phrase'}; ending=q(kw['end_phrase'])
            templates=[f'End your answer with the exact phrase {ending}.',
                       f'The final words of your response must be exactly {ending}.',
                       f'Required ending (nothing should appear after it): {ending}.']
        else: raise ValueError(instruction_id)
        return dict(zip(('P1','P2','P3'),templates))[variant]

    def new_rule(typ,kw,v):
        a=kwargs_one(kw);q=lambda z:json.dumps(str(z),ensure_ascii=False)
        assert v in R_TEMPLATES
        if typ=='keywords:forbidden_words':
            ws=', '.join(q(x) for x in a['forbidden_words'])
            return (f'Forbidden vocabulary for the final answer: {ws}. Do not use any of these terms.' if v=='R1' else
                    f'Ensure that none of the following expressions appears in your reply: {ws}.')
        if typ=='keywords:existence':
            ws=', '.join(q(x) for x in a['keywords'])
            return (f'Mandatory terms to include in the output: {ws}. Include every term.' if v=='R1' else
                    f'Your complete answer must include all these exact words: {ws}.')
        if typ=='keywords:frequency':
            n=int(a['frequency']);w=q(a['keyword']);rel=a['relation']
            assert rel in ('at least','less than')
            if rel=='at least':
                return (f'Occurrence requirement: include the word {w} a minimum of {n} times.' if v=='R1' else
                        f'The word {w} must be present {n} times or more in the complete reply.')
            return (f'Occurrence restriction: use the word {w} fewer than {n} times.' if v=='R1' else
                    f'The total number of uses of the word {w} must be strictly below {n}.')
        if typ=='detectable_content:number_placeholders':
            n=int(a['num_placeholders'])
            return (f'Square-bracket placeholders are required: add at least {n} nonempty entries such as [name].' if v=='R1' else
                    f'Include {n} or more placeholders written within square brackets, for example [name].')
        if typ=='startend:end_checker':
            end=q(a['end_phrase'])
            return (f'Mandatory last phrase: {end}. The response must finish with that exact phrase.' if v=='R1' else
                    f'The exact final words of your answer must be {end}, without any subsequent text.')
        raise ValueError(typ)

    def make_prompt(task,rule,first=False):
        
        
        return (str(rule)+'\n'+str(task)) if first else (str(task)+'\n'+str(rule))

In [ ]:
if not REPLAY_COMPLETE:
    
    rows=[]
    for task in range(102):
        for typ in PRIMARY_TYPES:
            tid=TYPE_TO_ID[typ]
            x=raw[tid*102+task]
            kw=x['kwargs']
            for variant in ('original','P1','P2','P3','R1','R2'):
                if variant=='original':prompt=x['prompt']
                elif variant in ('P1','P2','P3'):
                    prompt=base_tasks[task]+'\n'+phrase(typ,kwargs_one(kw),variant)
                else:prompt=make_prompt(base_tasks[task],new_rule(typ,kw,variant))
                rows.append({'id':f'{task:03d}_{tid}_{variant}','task_id':task,'type_id':tid,
                            'instruction_type':typ,'variant':variant,'prompt':prompt,
                            'kwargs':kw,'task_text':base_tasks[task]})
    assert len(rows)==102*3*6 and len({r['id'] for r in rows})==len(rows)
    assert len({r['prompt'] for r in rows})==len(rows),'Source prompts duplicated unexpectedly'
    records=pd.DataFrame(rows)
    TASKS=np.arange(102)
    outer=list(KFold(n_splits=FOLDS,shuffle=True,random_state=SEED).split(TASKS))
    fold_task={int(TASKS[j]):i for i,(_,te) in enumerate(outer) for j in te}
    assert len(fold_task)==102
    PROTOCOL={
     'study':'ifeval_simple_corrected_phase3_cross_model_v1', 'model':MODEL,
     'model_id':MODEL_ID,'revision':MODEL_REV,'dataset':SOURCE,
     'seed':SEED,'folds':FOLDS,'fold_assignment':{str(k):int(v) for k,v in fold_task.items()},
     'layers':list(LAYERS),'primary_layer':PRIMARY_LAYER,
     'types':list(PRIMARY_TYPES),'type_universe':TYPES,'training_variants':['original','P1','P2','P3'],
     'test_variants':['R1','R2'],'Cs':list(Cs),'penalties':list(DELTA_PENALTIES),
     'max_new_tokens':GENERATION_BUDGET,'max_input_tokens':MAX_INPUT_TOKENS,
     'decoding':'greedy','attention':'sdpa','dtype':'bf16',
     'transformers_version':transformers.__version__,'torch_version':torch.__version__,
     'gpu_name':GPU_NAME, 'chat_template':'single_user_add_generation_prompt',
     'prompt_manifest_sha256':sha(canonical([{'id':r['id'],'prompt':r['prompt'],'kwargs':r['kwargs']} for r in rows]).encode())
    }
    RUN_HASH=sha(canonical(PROTOCOL).encode())[:20]
    RUN=MODEL_ROOT/'runs'/RUN_HASH
    RUN.mkdir(parents=True,exist_ok=True)
    freeze_json(RUN/'protocol.json',PROTOCOL)
    atomic_json(RUN/'source_provenance.json',SOURCE)
    records.to_json(RUN/'frozen_prompts.jsonl',orient='records',lines=True,force_ascii=False)
    GEN_DIR=RUN/'generations';ST_DIR=RUN/'prompt_states';P_DIR=RUN/'probes';RESULT_DIR=RUN/'results'
    for d in (GEN_DIR,ST_DIR,P_DIR,RESULT_DIR):d.mkdir(parents=True,exist_ok=True)
    print('Run folder:',RUN)
    print('Prompt count:',len(records),'| expected original',len(records[records.variant.eq('original')]))

### 1.2 Frozen Phi-3 generations and pre-generation hidden states

Resumable prompt-level caches. Completed entries are verified against the original checker and are not silently regenerated.

In [ ]:
if not REPLAY_COMPLETE:
    
    model=None;tokenizer=None;INPUT_DEVICE=None;HIDDEN_SIZE=int(CFG['expected_hidden'])
    from transformers import AutoModelForCausalLM,AutoTokenizer

    def ensure_model():
        global model,tokenizer,INPUT_DEVICE,HIDDEN_SIZE
        if model is not None:return
        assert transformers.__version__=='4.53.3'
        tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REV,trust_remote_code=False)
        model=AutoModelForCausalLM.from_pretrained(
            MODEL_ID,revision=MODEL_REV,torch_dtype=DTYPE,device_map='auto',
            trust_remote_code=False,attn_implementation='sdpa')
        model.eval()
        assert model.config.hidden_size==CFG['expected_hidden'], (model.config.hidden_size,CFG['expected_hidden'])
        assert len(model.model.layers)==CFG['expected_layers'],len(model.model.layers)
        assert model.__class__.__module__.startswith('transformers.models.'+CFG['family']),model.__class__.__module__
        assert getattr(model.config,'_commit_hash',None) in (None,MODEL_REV), 'Loaded model revision does not match pin'
        INPUT_DEVICE=model.get_input_embeddings().weight.device
        assert INPUT_DEVICE.type=='cuda'
        HIDDEN_SIZE=int(model.config.hidden_size)
        print('Loaded',MODEL_ID,'@',MODEL_REV,'| hidden width',HIDDEN_SIZE)

    def encode_prompt(p):
        assert tokenizer is not None
        enc=tokenizer.apply_chat_template([{'role':'user','content':p}],
            add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors='pt')
        n=int(enc['input_ids'].shape[-1])
        assert 0<n<=MAX_INPUT_TOKENS, f'Input length {n} exceeds MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}; never silently truncate'
        return {k:v.to(INPUT_DEVICE) for k,v in enc.items()},n

    @torch.inference_mode()
    def generate_response(p):
        enc,n=encode_prompt(p)
        tokens=model.generate(**enc,max_new_tokens=GENERATION_BUDGET,do_sample=False,
                              pad_token_id=tokenizer.eos_token_id,use_cache=True)
        new=tokens[0,n:]
        k=int(new.numel())
        ends={int(tokenizer.eos_token_id)}
        if getattr(tokenizer,'additional_special_tokens_ids',None):
            ends.update(int(x) for x in tokenizer.additional_special_tokens_ids)
        cap=bool(k>=GENERATION_BUDGET and (k==0 or int(new[-1]) not in ends))
        response=tokenizer.decode(new,skip_special_tokens=True).strip()
        del tokens,enc,new
        return {'response':response,'input_tokens':n,'output_tokens':k,'hit_token_cap':cap}

    @torch.inference_mode()
    def extract_state(p):
        enc,_=encode_prompt(p)
        out=model(**enc,output_hidden_states=True,use_cache=False,return_dict=True)
        v=np.stack([out.hidden_states[i][0,-1].float().cpu().numpy() for i in LAYERS]).astype(np.float16)
        del out,enc
        assert v.shape==(len(LAYERS),HIDDEN_SIZE) and np.isfinite(v).all()
        return v

    
    def cache_key(r):return sha((str(r['id'])+'|'+sha(r['prompt'].encode())).encode())[:32]
    def expected_args(r):
        return {'id':str(r['id']),'prompt_sha256':sha(r['prompt'].encode()),
                'rule_sha256':sha(canonical(kwargs_one(r['kwargs'])).encode()),
                'run_sha256':sha(canonical(PROTOCOL).encode()),'model_id':MODEL_ID,
                'revision':MODEL_REV,'max_new_tokens':GENERATION_BUDGET,'layers':list(LAYERS)}

    def get_generation(r):
        p=GEN_DIR/(cache_key(r)+'.json');args=expected_args(r)
        if p.is_file():
            item=json.loads(p.read_text(encoding='utf-8'))
            assert item['args']==args,f'Generation fingerprint changed: {p}'
            assert int(item['label'])==strict_score(r,item['response']),'Cached verifier label mismatch'
            return item,True
        ensure_model()
        generated=generate_response(r['prompt'])
        item={'args':args,**generated,'label':strict_score(r,generated['response'])}
        atomic_json(p,item)
        return item,False

    def get_state(r):
        key=cache_key(r);p=ST_DIR/(key+'.npz');meta=ST_DIR/(key+'.json');args=expected_args(r)
        if p.is_file() and meta.is_file():
            assert json.loads(meta.read_text(encoding='utf-8'))==args,'State fingerprint mismatch'
            with np.load(p,allow_pickle=False) as z:a=z['states'].astype(np.float32)
            assert a.shape==(len(LAYERS),HIDDEN_SIZE) and np.isfinite(a).all()
            return a,True
        ensure_model()
        a=extract_state(r['prompt'])
        atomic_npz(p,states=a)
        atomic_json(meta,args)
        return a.astype(np.float32),False

    
    
    assert PRIMARY_LAYER in LAYERS

In [ ]:
if not REPLAY_COMPLETE:
    
    
    rows_scored=[];states=[];reused_gen=reused_states=0
    for k,r in enumerate(tqdm(rows,desc=MODEL+' Simple prompts')):
        gen,hit=get_generation(r)
        z,hit_state=get_state(r)
        reused_gen+=int(hit);reused_states+=int(hit_state)
        rows_scored.append({**r,'label':int(gen['label']),
                 'response':gen['response'],'input_tokens':int(gen['input_tokens']),
                 'output_tokens':int(gen['output_tokens']),'hit_token_cap':bool(gen['hit_token_cap']),
                 'fold':fold_task[int(r['task_id'])]})
        states.append(z)
        if (k+1)%50==0:
            print('Processed',k+1,'/',len(rows),'| cached responses',reused_gen,'| cached states',reused_states)
        if (k+1)%125==0:
            gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()
    scored=pd.DataFrame(rows_scored)
    state_array=np.stack(states).astype(np.float16)
    assert len(scored)==102*len(PRIMARY_TYPES)*6 and scored.id.is_unique
    assert state_array.shape==(len(scored),len(LAYERS),HIDDEN_SIZE)
    scored.to_json(RUN/'all_1836_scored.jsonl',orient='records',lines=True,force_ascii=False)
    np.save(RUN/'states_ordered.npy',state_array,allow_pickle=False)
    assert all(int(r.label)==strict_score(r._asdict(),r.response) for r in scored.itertuples(index=False)), 'Official checker replay mismatch'
    print('All outputs verified:',len(scored),'| generations reused',reused_gen,'| states reused',reused_states)
    print('Token cap rate:',round(float(scored.hit_token_cap.mean()),4))
    display(scored.groupby(['instruction_type','variant']).agg(n=('label','size'),success=('label','mean'),cap=('hit_token_cap','mean')).round(4))
    
    if model is not None:
        del model,tokenizer
        model=None;tokenizer=None;INPUT_DEVICE=None
        gc.collect();torch.cuda.empty_cache()

### 1.3 B0 / B2 / MT0 / MT2 fitting

Original weighted logistic objective, nested validation, and five task-disjoint folds. Held-out R1/R2 labels never enter fitting or hyperparameter selection.

In [ ]:
if not REPLAY_COMPLETE:
    
    train_df=scored[scored.variant.isin(['original','P1','P2','P3'])].copy().reset_index(drop=True)
    train_idx=scored.index[scored.variant.isin(['original','P1','P2','P3'])].to_numpy()
    X_old=state_array[train_idx,LAYERS.index(PRIMARY_LAYER)].astype(np.float32)
    assert len(train_df)==102*len(PRIMARY_TYPES)*4 and X_old.shape==(len(train_df),HIDDEN_SIZE)
    assert sorted(train_df.variant.unique().tolist())==['P1','P2','P3','original']
    print('Model-specific labels and hidden states ready for original Phase 3 training:',X_old.shape)

In [ ]:
if not REPLAY_COMPLETE:
    class Probe:
        def __init__(self,mean,scale,w,delta,offset,b,method,setting):
            self.mean=np.asarray(mean,np.float32);self.scale=np.asarray(scale,np.float32)
            self.w=np.asarray(w,np.float32);self.delta=np.asarray(delta,np.float32)
            self.offset=np.asarray(offset,np.float32);self.b=float(b)
            self.method=method;self.setting=setting
        def predict(self,X,t):
            X=np.asarray(X,np.float32)
            if X.ndim==1:X=X[None,:]
            t=np.asarray(t,dtype=int).reshape(-1)
            assert len(X)==len(t) and X.shape[1]==len(self.w)
            v=(X-self.mean)/self.scale
            logits=np.sum(v*(self.w[None,:]+self.delta[t]),axis=1)+self.b+self.offset[t]
            return expit(np.clip(logits,-35,35))
        def save(self,path):
            atomic_npz(path,mean=self.mean,scale=self.scale,w=self.w,delta=self.delta,
                       offset=self.offset,b=np.array(self.b),setting=np.array(canonical(self.setting)))
        @staticmethod
        def load(path,method):
            with np.load(path,allow_pickle=False) as z:
                return Probe(z['mean'],z['scale'],z['w'],z['delta'],z['offset'],
                             float(z['b']),method,json.loads(str(z['setting'])))

    def fit_probe(task_ids,method,C,penalty=16.,max_iter=80):
        assert method in METHODS
        subset=train_df.task_id.isin(task_ids) & (train_df.variant.eq('original') if method in ('B0','MT0') else True)
        idx=np.flatnonzero(subset.to_numpy())
        orig_idx=np.flatnonzero((train_df.task_id.isin(task_ids)&train_df.variant.eq('original')).to_numpy())
        scaler=StandardScaler().fit(X_old[orig_idx])
        scale=np.maximum(scaler.scale_,1e-6).astype(np.float32)
        X=((X_old[idx]-scaler.mean_)/scale).astype(np.float32)
        y=train_df.iloc[idx].label.to_numpy(dtype=np.float32)
        t=train_df.iloc[idx].type_id.to_numpy(dtype=np.int64)
        assert np.unique(y).size==2
        groups=train_df.iloc[idx].groupby(['task_id','type_id']).size().to_dict()
        sample_weights=np.array([1/groups[(int(a),int(b))] for a,b in train_df.iloc[idx][['task_id','type_id']].to_numpy()],dtype=np.float32)
        sample_weights/=sample_weights.sum()
        
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        xt=torch.from_numpy(X).to(device);yt=torch.from_numpy(y).to(device)
        tt=torch.from_numpy(t).to(device);sw=torch.from_numpy(sample_weights).to(device)
        ww=torch.nn.Parameter(torch.zeros(HIDDEN_SIZE,device=device,dtype=torch.float32))
        bb=torch.nn.Parameter(torch.zeros((),device=device,dtype=torch.float32))
        mt=method.startswith('MT')
        dd=torch.nn.Parameter(torch.zeros((N_TYPES,HIDDEN_SIZE),device=device,dtype=torch.float32)) if mt else None
        oo=torch.nn.Parameter(torch.zeros(N_TYPES,device=device,dtype=torch.float32)) if mt else None
        params=[ww,bb]+([dd,oo] if mt else [])
        opt=torch.optim.LBFGS(params,lr=1.,max_iter=max_iter,line_search_fn='strong_wolfe',
                              tolerance_grad=2e-5,tolerance_change=1e-9)
        unit_count=int(len(groups));ridge=1.0/(2*float(C)*unit_count)
        def closure():
            opt.zero_grad()
            logit=(xt* (ww[None,:]+dd[tt] if mt else ww[None,:])).sum(dim=1)+bb
            if mt:logit=logit+oo[tt]
            bce=torch.nn.functional.binary_cross_entropy_with_logits(logit,yt,reduction='none')
            loss=(sw*bce).sum()+ridge*ww.square().sum()
            if mt:loss=loss+ridge*float(penalty)*dd.square().sum()+ridge*oo.square().sum()
            assert torch.isfinite(loss).item(),'Nonfinite probe loss'
            loss.backward()
            return loss
        opt.step(closure)
        with torch.no_grad():
            out=Probe(scaler.mean_,scale,ww.cpu().numpy(),
               dd.cpu().numpy() if mt else np.zeros((N_TYPES,HIDDEN_SIZE),dtype=np.float32),
               oo.cpu().numpy() if mt else np.zeros(N_TYPES,dtype=np.float32),
               bb.cpu().item(),method,{'C':float(C),'penalty':float(penalty)})
        del xt,yt,tt,sw,ww,bb,dd,oo,opt
        return out

    def macro_old(probe,task_ids,variants):
        scores=[]
        for typ in PRIMARY_TYPES:
            tid=TYPE_TO_ID[typ]
            for variant in variants:
                ix=np.flatnonzero((train_df.task_id.isin(task_ids)&train_df.instruction_type.eq(typ)&train_df.variant.eq(variant)).to_numpy())
                if len(ix)<1:continue
                y=train_df.iloc[ix].label.to_numpy(dtype=int)
                if len(np.unique(y))<2:continue
                pred=probe.predict(X_old[ix],np.full(len(ix),tid))
                scores.append(auc(y,pred))
        return float(np.mean(scores)) if scores else float('nan')

In [ ]:
if not REPLAY_COMPLETE:
    
    
    PROBE_DIR=P_DIR
    PROBE_DIR.mkdir(exist_ok=True)

    def selected_rows(task_ids,variants):
        return np.flatnonzero((train_df.task_id.isin(task_ids)&train_df.variant.isin(variants)).to_numpy())

    
    train_content=sha(canonical([{'id':r['id'],'label':int(r['label']),
          'prompt_sha256':sha(str(r['prompt']).encode())} for r in train_df.to_dict('records')]).encode())
    fit_def={'protocol':RUN_HASH,'training_sha256':train_content,
             'methods':list(METHODS),'source_method':'Phase3_torch_LBFGS_weighted_BCE',
             'hidden_width':HIDDEN_SIZE,'type_count':N_TYPES,
             'fit_iters_train':80,'fit_iters_refit':110}
    freeze_json(PROBE_DIR/'fit_manifest.json',fit_def)
    FIT_FP=sha(canonical(fit_def).encode())
    fitted={};tuning=[]
    for fold,(trval_idx,test_idx) in enumerate(outer):
        trval=TASKS[trval_idx];test=TASKS[test_idx]
        inner_train,inner_val=train_test_split(trval,test_size=0.25,random_state=SEED+fold)
        assert set(inner_train).isdisjoint(inner_val) and set(test).isdisjoint(trval)
        base_orig=None
        for method in METHODS:
            mpath=PROBE_DIR/f'fold{fold}_{method}.npz'
            meta=PROBE_DIR/f'fold{fold}_{method}.json'
            if mpath.is_file() and meta.is_file():
                setting=json.loads(meta.read_text(encoding='utf-8'))
                assert setting['fit_fp']==FIT_FP, f'Fitted readout fingerprint mismatch: {mpath}'
                fitted[(fold,method)]=Probe.load(mpath,method)
                tuning.append(setting)
                if method=='B0':base_orig=setting['val_original_auc']
                print(f'Reused fold={fold} method={method}')
                continue
            choices=[(C,16.) for C in Cs] if not method.startswith('MT') else list(itertools.product(Cs,DELTA_PENALTIES))
            candidates=[]
            for C,penalty in choices:
                p=fit_probe(inner_train,method,C,penalty)
                valid=macro_old(p,inner_val,('P1','P2','P3'))
                org=macro_old(p,inner_val,('original',))
                if np.isfinite(valid) and np.isfinite(org):candidates.append((valid,org,C,penalty))
            if not candidates:
                raise RuntimeError(f'Model {MODEL}, fold {fold}, {method}: no binary-class validation cells. '                               'Report class support; do not silently tune on R1/R2.')
            feasible=[x for x in candidates if base_orig is None or x[1]>=base_orig-ORIG_MARGIN]
            best=max(feasible if feasible else candidates,key=lambda x:(x[0],x[1],-x[2],-x[3]))
            final=fit_probe(trval,method,best[2],best[3],max_iter=110)
            setting={'fold':fold,'method':method,'fit_fp':FIT_FP,
                     'trainval_tasks':list(map(int,trval)),'test_tasks':list(map(int,test)),
                     'C':float(best[2]),'delta_penalty':float(best[3]),
                     'val_shift_auc':float(best[0]),'val_original_auc':float(best[1]),
                     'val_feasible':bool(feasible)}
            final.save(mpath);atomic_json(meta,setting)
            fitted[(fold,method)]=final;tuning.append(setting)
            if method=='B0':base_orig=best[1]
            print(f'FITTED fold={fold} method={method}')
    assert len(fitted)==FOLDS*len(METHODS)
    tune_df=pd.DataFrame(tuning)
    tune_df.to_csv(RESULT_DIR/'probe_tuning.csv',index=False)
    print('All',len(fitted),'fold-specific probes are fitted and frozen from this model.')

### 1.4 Primary transfer results, paired changes, and controls

Keep original and held-out AUROC, the MT2−B2 comparison, grouped confidence intervals, original-domain retention, per-type scores, text-only/type-prior controls, and individual flip tracking.

In [ ]:
if not REPLAY_COMPLETE:
    
    all_eval=[]
    for j,r in enumerate(scored.itertuples(index=False)):
        if r.variant not in ('original','R1','R2'):continue
        fold=fold_task[int(r.task_id)]
        p={'id':r.id,'task_id':int(r.task_id),'fold':fold,'type':r.instruction_type,
           'variant':r.variant,'label':int(r.label),'capped':bool(r.hit_token_cap),
           'input_tokens':int(r.input_tokens),'output_tokens':int(r.output_tokens)}
        for m in METHODS:
            p[m]=float(fitted[(fold,m)].predict(
                 state_array[j,LAYERS.index(PRIMARY_LAYER)].astype(np.float32),[int(r.type_id)])[0])
        
        old_tr=train_df[(train_df.task_id.isin(TASKS[outer[fold][0]]))&
                        train_df.instruction_type.eq(r.instruction_type)&train_df.variant.eq('original')]
        p['type_only']=float((int(old_tr.label.sum())+1)/(len(old_tr)+2))
        all_eval.append(p)
    preds=pd.DataFrame(all_eval)
    assert len(preds)==102*len(PRIMARY_TYPES)*3
    assert preds.groupby(['task_id','type']).size().eq(3).all()
    assert all(x not in set(TASKS[outer[int(p.fold)][0]]) for p in preds.itertuples(index=False) for x in [p.task_id])
    preds.to_csv(RESULT_DIR/'oof_original_and_R_predictions.csv',index=False)

    def macro(t,m,variants=('R1','R2')):
        scores=[]
        for typ in PRIMARY_TYPES:
            for v in variants:
                g=t[(t.type==typ)&(t.variant==v)]
                if len(g) and g.label.nunique()==2:scores.append(safe_auc(g.label,g[m]))
        return float(np.mean(scores)) if scores else np.nan

    metrics=[]
    for (typ,v),g in preds.groupby(['type','variant']):
        for m in (*METHODS,'type_only'):
            metrics.append({'type':typ,'variant':v,'method':m,'n':len(g),
                'positives':int(g.label.sum()),'negatives':int(len(g)-g.label.sum()),
                'auroc':safe_auc(g.label,g[m]),
                'brier':float(brier_score_loss(g.label,g[m])),
                'logloss':float(log_loss(g.label,g[m],labels=[0,1]))})
    metrics_df=pd.DataFrame(metrics)
    metrics_df.to_csv(RESULT_DIR/'per_type_formulation_metrics.csv',index=False)
    print('Original-domain and held-out AUROC:')
    display(metrics_df.groupby(['variant','method']).auroc.mean().unstack('method').round(4))

In [ ]:
if not REPLAY_COMPLETE:
    
    
    R_pred=preds[preds.variant.isin(['R1','R2'])].copy()
    oof=preds[preds.variant.eq('original')].copy()
    old_lookup={(int(r.task_id),r.type):r for r in oof.itertuples(index=False)}
    flip_rows=[]
    for r in R_pred.itertuples(index=False):
        old=old_lookup[(int(r.task_id),r.type)]
        change=int(r.label)-int(old.label)
        row={'task_id':int(r.task_id),'type':r.type,'variant':r.variant,
             'old_label':int(old.label),'new_label':int(r.label),'change':change,
             'old_capped':bool(old.capped),'new_capped':bool(r.capped)}
        for m in METHODS:
            delta=float(getattr(r,m)-getattr(old,m))
            row[m+'_delta']=delta
            row[m+'_correct']=float(delta*change>0) if change else np.nan
        flip_rows.append(row)
    flip_df=pd.DataFrame(flip_rows)
    flip_df.to_csv(RESULT_DIR/'paired_behavioral_flips.csv',index=False)

    H1=[]
    for typ in PRIMARY_TYPES:
        for v in R_TEMPLATES:
            g=flip_df[(flip_df.type==typ)&flip_df.variant.eq(v)]
            H1.append({'type':typ,'variant':v,'n':len(g),'flips':int(g.change.ne(0).sum()),
                       'flip_rate':float(g.change.ne(0).mean()),
                       'up':int(g.change.eq(1).sum()),'down':int(g.change.eq(-1).sum()),
                       'mean_success_change':float(g.change.mean()),
                       'mcnemar_exact_p':float(binomtest(int(g.change.eq(1).sum()),int(g.change.ne(0).sum()),0.5).pvalue)
                            if int(g.change.ne(0).sum()) else 1.0})
    H1=pd.DataFrame(H1)
    H1.to_csv(RESULT_DIR/'H1_pairwise_behavior.csv',index=False)

    def family_bootstrap(table, variants, seed_offset):
        """Fast source-family bootstrap with exact weighted tie-aware AUROC.

        Resampling 102 task families with replacement is equivalent to giving
        each original family's observations their multinomial sample count.
        All methods use identical draws (paired effects). Single-class draw
        cells are skipped, matching the source Phase 3 macro definition.
        """
        rng=np.random.default_rng(SEED+seed_offset)
        tasks=np.array(sorted(table.task_id.unique()))
        pos={int(k):i for i,k in enumerate(tasks)}
        draws=rng.multinomial(len(tasks),np.full(len(tasks),1./len(tasks)),size=N_BOOT)
        methods=(*METHODS,'type_only')
        auc_boot={m:np.full(N_BOOT,np.nan) for m in methods}
        brier_boot={m:np.full(N_BOOT,np.nan) for m in methods}
        groups=[]
        for typ in PRIMARY_TYPES:
            for v in variants:
                g=table[(table.type==typ)&table.variant.eq(v)]
                if g.empty:continue
                ix=np.array([pos[int(t)] for t in g.task_id],dtype=np.int32)
                y=g.label.to_numpy(dtype=int)
                scores={m:g[m].to_numpy(dtype=float) for m in methods}
                ties={m:np.unique(scores[m],return_inverse=True)[1] for m in methods}
                groups.append((ix,y,scores,ties))
        for m in methods:
            for k,draw in enumerate(draws):
                aa=[];bb=[]
                for ix,y,scores,ties in groups:
                    w=draw[ix].astype(float)
                    if w.sum()==0:continue
                    p=scores[m]
                    bb.append(np.dot(w,(y-p)**2)/w.sum())
                    posw=w*y;negw=w*(1-y)
                    if posw.sum()==0 or negw.sum()==0:continue
                    group_ids=ties[m]
                    ngrp=int(group_ids.max())+1
                    pg=np.bincount(group_ids,weights=posw,minlength=ngrp)
                    ng=np.bincount(group_ids,weights=negw,minlength=ngrp)
                    win=np.dot(pg,np.cumsum(ng)-ng+0.5*ng)
                    aa.append(float(win/(posw.sum()*negw.sum())))
                auc_boot[m][k]=float(np.mean(aa)) if aa else np.nan
                brier_boot[m][k]=float(np.mean(bb)) if bb else np.nan
        return auc_boot,brier_boot

    def interval(s,observed,one_sided=False):
        s=np.asarray(s,dtype=float);s=s[np.isfinite(s)]
        if len(s)==0:return {'delta':float(observed),'ci_low':np.nan,'ci_high':np.nan,'valid':0}
        ans={'delta':float(observed),'ci_low':float(np.quantile(s,.025)),
             'ci_high':float(np.quantile(s,.975)),'valid':int(len(s))}
        if one_sided:ans['one_sided_95_lower']=float(np.quantile(s,.05))
        return ans

    
    boot_R,boot_R_brier=family_bootstrap(R_pred,R_TEMPLATES,201)
    boot_O,boot_O_brier=family_bootstrap(oof,('original',),301)
    h2=[];id_ret=[];briers=[]
    for m in ('B2','MT0','MT2'):
        h2.append({'method':m,**interval(boot_R[m]-boot_R['B0'],macro(R_pred,m)-macro(R_pred,'B0'))})
        id_ret.append({'method':m,**interval(boot_O[m]-boot_O['B0'],
                           macro(oof,m,('original',))-macro(oof,'B0',('original',)),True)})
        briers.append({'method':m,**interval(boot_R_brier['B0']-boot_R_brier[m],
             float(np.mean([(g.B0.sub(g.label).pow(2).mean()-g[m].sub(g.label).pow(2).mean())
                            for _,g in R_pred.groupby(['type','variant'])])))})
    h2=pd.DataFrame(h2);ret=pd.DataFrame(id_ret);brier=pd.DataFrame(briers)
    ret['noninferior_at_2pt_onesided_95']=ret.one_sided_95_lower > -ORIG_MARGIN
    h2.to_csv(RESULT_DIR/'H2_paired_OOD_AUROC_bootstrap.csv',index=False)
    ret.to_csv(RESULT_DIR/'original_domain_retention.csv',index=False)
    brier.to_csv(RESULT_DIR/'H2_Brier_gain_bootstrap.csv',index=False)
    flip_acc=[]
    for typ in PRIMARY_TYPES:
        for v in R_TEMPLATES:
            g=flip_df[(flip_df.type==typ)&flip_df.variant.eq(v)&flip_df.change.ne(0)]
            for m in METHODS:
                flip_acc.append({'type':typ,'variant':v,'method':m,'n_flips':len(g),
                                 'correct_direction':float(g[m+'_correct'].mean()) if len(g) else np.nan})
    pd.DataFrame(flip_acc).to_csv(RESULT_DIR/'flip_prediction_diagnostic.csv',index=False)
    print('H1 wording flips:');display(H1.round(3))
    print('H2 held-out differences:');display(h2.round(4))
    print('Original-domain retention:');display(ret.round(4))

In [ ]:
if not REPLAY_COMPLETE:
    
    if RUN_TEXT_CONTROL:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        texts_old=train_df.prompt.astype(str).tolist()
        rt=scored[scored.variant.isin(['R1','R2'])].reset_index(drop=True)
        text_scores=np.zeros(len(rt),dtype=float)
        for fold,(tr_idx,te_idx) in enumerate(outer):
            train_tasks=TASKS[tr_idx]
            ix=np.flatnonzero((train_df.task_id.isin(train_tasks)&train_df.variant.eq('original')).to_numpy())
            ev=np.flatnonzero(rt.task_id.map(fold_task).eq(fold).to_numpy())
            vec=TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,max_features=6000,
                                sublinear_tf=True,lowercase=True)
            A=vec.fit_transform([texts_old[j] for j in ix])
            yt=train_df.iloc[ix].label.to_numpy(int)
            if np.unique(yt).size<2:raise RuntimeError('Single-class text-control training fold')
            clf=LogisticRegression(C=1.,max_iter=500,solver='liblinear',random_state=SEED)
            clf.fit(A,yt)
            text_scores[ev]=clf.predict_proba(vec.transform(rt.iloc[ev].prompt.astype(str)))[:,1]
        assert set(rt.id)==set(R_pred.id)
        txt=pd.DataFrame({'id':rt.id,'text_tfidf':text_scores})
        R_with_text=R_pred.merge(txt,on='id',validate='one_to_one')
        R_with_text.to_csv(RESULT_DIR/'heldout_predictions_with_text_control.csv',index=False)
        text_metrics=[]
        for (typ,v),g in R_with_text.groupby(['type','variant']):
            text_metrics.append({'type':typ,'variant':v,'char_tfidf_auc':safe_auc(g.label,g.text_tfidf),
                                 'MT2_auc':safe_auc(g.label,g.MT2)})
        pd.DataFrame(text_metrics).to_csv(RESULT_DIR/'char_tfidf_control.csv',index=False)
        print('Text vs hidden state macro AUROC:',macro(R_with_text,'text_tfidf'),macro(R_with_text,'MT2'))

### 1.5 Reproducibility export

Exports complete final-paper predictions and protocol fingerprints; omits exploratory intermediate-layer tests.

In [ ]:
if not REPLAY_COMPLETE:
    
    summary={
     'model':MODEL,'model_id':MODEL_ID,'revision_resolved_at_first_run':MODEL_REV,
     'task_families':102,'constraint_types':list(PRIMARY_TYPES),
     'generated_count':len(scored),'capped_count':int(scored.hit_token_cap.sum()),
     'ood_macro_auroc':{m:macro(R_pred,m) for m in METHODS},
     'o_macro_auroc':{m:macro(oof,m,('original',)) for m in METHODS},
     'mt2_minus_b0_ood_auc':float(macro(R_pred,'MT2')-macro(R_pred,'B0')),
     'wording_flip_rate':float(flip_df.change.ne(0).mean()),
     'source_sha256':SOURCE, 'protocol_sha256':sha(canonical(PROTOCOL).encode()),
     'semantic_note':'R1 placeholders says nonempty but official checker counts empty [] as placeholders; original Phase 3 exact text preserved.',
     'hardware_note':'Cache fingerprint contains GPU type to avoid mixing generation environments.',
     'readout_note':'All 20 model-specific probes fitted on generated original/P1-P3 labels from this run, task-disjoint.'}
    atomic_json(RESULT_DIR/'FINAL_SUMMARY.json',summary)
    files_for_export=[RUN/'protocol.json',RUN/'source_provenance.json',RUN/'frozen_prompts.jsonl',
                      RUN/'all_1836_scored.jsonl',RUN/'states_ordered.npy']
    files_for_export+=list(sorted(RESULT_DIR.rglob('*')))
    files_for_export+=list(sorted(P_DIR.glob('*')))
    files_for_export=[x for x in files_for_export if x.is_file()]
    manifest={str(p.relative_to(RUN)):shafile(p) for p in files_for_export}
    atomic_json(RUN/'EXPORT_SHA256_MANIFEST.json',manifest)
    result_zip=RUN/f'IFEvalSimple_{MODEL}_Full_Results.zip'
    with zipfile.ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for p in files_for_export:z.write(p,arcname=p.relative_to(RUN).as_posix())
        z.write(RUN/'EXPORT_SHA256_MANIFEST.json','EXPORT_SHA256_MANIFEST.json')
    with zipfile.ZipFile(result_zip) as z:
        assert z.testzip() is None
        for name,ref_hash in manifest.items():assert sha(z.read(name))==ref_hash,name
    print('ALL DONE. Results archive:',result_zip)
    print('Files:',len(manifest),'ZIP bytes:',result_zip.stat().st_size)
    print('H2 OOD macro AUROC:',summary['ood_macro_auroc'])
    print('MT2 minus B0:',summary['mt2_minus_b0_ood_auc'])
    print('Flip rate:',summary['wording_flip_rate'])
    print('Original root never modified:',PHI_LEGACY_ROOT/'runs')

In [ ]:
if not REPLAY_COMPLETE:
    
    import gc, torch
    if 'model' in globals() and model is not None:
        del model
        model = None
    if 'tokenizer' in globals() and tokenizer is not None:
        del tokenizer
        tokenizer = None
    gc.collect()
    torch.cuda.empty_cache()
    SIMPLE_RESULTS = str(RESULT_DIR)
    print('IFEval-Simple results:', SIMPLE_RESULTS)
    SIMPLE_CONTEXT={"fitted":fitted.copy(),"fold_task":fold_task.copy(),"types":tuple(PRIMARY_TYPES),"probe_dir":str(PROBE_DIR)}


# Part II. FollowBench-derived controlled instruction setting

**Source protocol:** official English FollowBench task evolution families, plus deterministic added verifiable constraints, with group-disjoint five-fold evaluation. The `..._Fixed` notebook is the source of truth for prompt generation and lineage checking. Outputs are scored against the **added IFEval-style constraint**, not against the original FollowBench task rubric. The existing 1,536-token output cap is preserved.

In [ ]:
if not REPLAY_COMPLETE:
    import os, sys, re, json, hashlib, itertools, math, random, shutil, importlib
    from pathlib import Path
    from collections import Counter, defaultdict
    import numpy as np, pandas as pd, requests, torch, nltk
    from tqdm.auto import tqdm
    from IPython.display import display
    from sklearn.model_selection import KFold, train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
    from sklearn.linear_model import LogisticRegression
    from scipy.special import expit
    from scipy.stats import binomtest

    FOLDER='ULM_Phi3_FollowBench_Controlled_v1'
    ROOT=ULM_WORKDIR/'FollowBench-derived'/FOLDER
    DATA=ROOT/'data'; CACHE=ROOT/'cache'; RESULTS=ROOT/'results'; MODELS=ROOT/'probes'
    for d in (DATA,CACHE,RESULTS,MODELS):d.mkdir(parents=True,exist_ok=True)

    SEED=20261018
    MODEL_ID='microsoft/Phi-3-mini-128k-instruct'
    MODEL_REV='f3c06aed622e14ca0abf5115094e4fc9a9948f36'
    LAYERS=(14,26,32); PRIMARY_LAYER=14; FOLDS=5
    GEN_BUDGET=1536; MAX_INPUT_TOKENS=4096
    N_PILOT_GROUPS=30; MIN_STUDY_GROUPS=85; MAX_STUDY_GROUPS=160
    PILOT_MIN_CLASS=5; PILOT_MAX_CAP=.12; PRIMARY_N_TYPES=3
    MIN_TEST_CLASS_PER_FORM=10
    BOOTSTRAPS=2000; Cs=(.03,.1,.3); DELTA_PENALTIES=(4.,16.)
    ID_MARGIN=.02; METHODS=('B0','B2','MT0','MT2')
    RUN_FULL_541=False
    RUN_TEXT_CONTROL=True
    RUN_GEOMETRY=True  
    RUN_OTHER_LAYERS=False

    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    def sha(b):return hashlib.sha256(b).hexdigest()
    def shafile(p):
        h=hashlib.sha256()
        with open(p,'rb') as f:
            for block in iter(lambda:f.read(1<<20),b''):h.update(block)
        return h.hexdigest()
    def canon(x):return json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(',',':'))
    def save_json(p,x):
        p=Path(p);p.parent.mkdir(exist_ok=True,parents=True);tmp=p.with_name(p.name+'.tmp')
        tmp.write_text(json.dumps(x,indent=2,ensure_ascii=False,allow_nan=False,default=str),encoding='utf8');os.replace(tmp,p)
    def save_npz(p,**arrays):
        p=Path(p);p.parent.mkdir(exist_ok=True,parents=True);tmp=p.with_name(p.name+'.tmp')
        with open(tmp,'wb') as f:np.savez_compressed(f,**arrays)
        os.replace(tmp,p)
    def freeze(p,x):
        if Path(p).exists():
            assert json.loads(Path(p).read_text())==json.loads(json.dumps(x,sort_keys=True,default=str)), f'Configuration mismatch: {p}. Choose a NEW FOLDER.'
        else:save_json(p,x)
    def auc(y,p):
        y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
        return float(roc_auc_score(y,p)) if len(y)>1 and np.unique(y).size==2 else np.nan
    def holm(ps):
        arr=np.asarray(ps,float);order=np.argsort(arr);out=np.ones(len(arr));v=0.
        for j,i in enumerate(order):v=max(v,min(1.,(len(arr)-j)*arr[i]));out[i]=v
        return out

    def fetch(url,path):
        path=Path(path)
        if path.is_file() and path.stat().st_size:return
        response=requests.get(url,timeout=120);response.raise_for_status()
        assert len(response.content)>50, f'Unexpected short download: {url}'
        tmp=path.with_name(path.name+'.part');tmp.write_bytes(response.content);os.replace(tmp,path)
    print('Workspace:',ROOT)

### 2.1 FollowBench task families and controlled verifiable constraints

In [ ]:
if not REPLAY_COMPLETE:
    
    FB_FILES=('content','situation','style','format','example','mixed')
    FB_BASE='https://raw.githubusercontent.com/YJiangcm/FollowBench/master/data/'
    FB_BACKUP='https://huggingface.co/datasets/YuxinJiang/FollowBench/resolve/main/data_en/'
    FB_DIR=DATA/'followbench_en';FB_DIR.mkdir(exist_ok=True)
    fb_source_files={}
    for cat in FB_FILES:
        fn=f'{cat}_constraints.json';p=FB_DIR/fn
        if not p.exists():
            try:fetch(FB_BASE+fn,p)
            except requests.RequestException:fetch(FB_BACKUP+fn,p)
        fb_source_files[cat]=p
        assert isinstance(json.loads(p.read_text(encoding='utf8')),list),f'Unexpected {fn} format'

    raw_entries=[]
    for cat,p in fb_source_files.items():
        dat=json.loads(p.read_text(encoding='utf8'))
        for item in dat:
            assert isinstance(item,dict),f'Invalid row in {p}'
            assert {'example_id','level','instruction'}.issubset(item),f'Missing FollowBench columns in {p}'
            ins=item['instruction']
            if not isinstance(ins,str):continue
            raw_entries.append({'category':cat,'example_id':str(item['example_id']),
                'level':int(item['level']),'instruction':ins,'source':str(item.get('source','')),
                'target':item.get('target','')})
    assert len(raw_entries)>300, 'Unexpected FollowBench row count; inspect downloads.'
    assert not pd.DataFrame(raw_entries).duplicated(['category','example_id','level']).any(), 'Duplicate source evolution-level key'

    
    VERIFIER_DIR=DATA/'instruction_following_eval';VERIFIER_DIR.mkdir(exist_ok=True)
    (VERIFIER_DIR/'__init__.py').touch()
    VER_FILES=('instructions.py','instructions_util.py','instructions_registry.py','evaluation_lib.py')
    V_URL='https://raw.githubusercontent.com/google-research/google-research/master/instruction_following_eval/'
    for fn in VER_FILES:fetch(V_URL+fn,VERIFIER_DIR/fn)
    verifier_sha={f:shafile(VERIFIER_DIR/f) for f in VER_FILES}
    for fn,prefix in {'instructions.py':'60e086f5342a','instructions_util.py':'a73797261eee',
                      'instructions_registry.py':'ec92d72c264f'}.items():
        assert verifier_sha[fn].startswith(prefix),f'Verifier changed: {fn}; audit before proceeding'
    if str(DATA) not in sys.path:sys.path.insert(0,str(DATA))
    from instruction_following_eval import instructions_registry
    for pkg in ('punkt','punkt_tab'):
        assert nltk.download(pkg,quiet=True),f'NLTK unavailable: {pkg}'

    def score_one(typ,kw,response,prompt):
        if not isinstance(response,str) or not response.strip():return 0
        obj=instructions_registry.INSTRUCTION_DICT[typ](typ)
        obj.build_description(**kw)
        args=obj.get_instruction_args()
        if args and 'prompt' in args:obj.build_description(prompt=prompt)
        return int(bool(obj.check_following(response)))
    assert score_one('punctuation:no_comma',{},'Hello world','x')==1
    assert score_one('punctuation:no_comma',{},'Hello, world','x')==0

    
    base_families=defaultdict(list)
    for row in raw_entries:
        if row['instruction'].strip():base_families[(row['category'],row['example_id'])].append(row)
    
    
    low_priority={2:0,1:1,3:2,4:3,5:4}
    high_priority={4:0,5:1,3:2,2:3,1:4}
    raw=[]
    for (cat,eid),rows in sorted(base_families.items()):
        valid=[r for r in rows if 45<=len(r['instruction'])<=4800 and r['level']>=1]
        if not valid:continue
        low=min(valid,key=lambda r:(low_priority.get(r['level'],99),len(r['instruction'])))
        high=min(valid,key=lambda r:(high_priority.get(r['level'],99),len(r['instruction'])))
        chosen=[low]+([high] if high['level']!=low['level'] else [])
        for choice in chosen:
            raw.append({'key':len(raw),'prompt':choice['instruction'],'category':cat,
                        'example_id':eid,'level':choice['level'],'source':choice['source'],
                        'family':f'{cat}:{eid}'})
    original_by_key={r['key']:r for r in raw}
    assert len(raw)>0
    pd.DataFrame(raw).to_csv(DATA/'native_family_representatives.csv',index=False)
    summary=pd.DataFrame(raw).groupby(['category','level']).size().rename('n').reset_index()
    display(summary)
    print('Native rows:',len(raw_entries),'| evolution families:',len(base_families),
          '| selected source-level prompts:',len(raw))
    print('Native constraints are NOT automatically graded in this study.')

In [ ]:
if not REPLAY_COMPLETE:
    
    
    import json, re
    import numpy as np
    from langdetect import DetectorFactory
    DetectorFactory.seed = SEED

    FORMS = ('original', 'P1', 'P2', 'R1', 'R2')
    CANDIDATE_TYPES = (
        'keywords:forbidden_words',
        'detectable_content:number_placeholders',
        'startend:end_checker',
        'change_case:english_lowercase',
        'change_case:english_capital',
        'punctuation:no_comma',
    )
    TYPE_TO_KW = {
        'detectable_content:number_placeholders': {'num_placeholders': 3},
        'startend:end_checker': {'end_phrase': 'This concludes my response.'},
        'change_case:english_lowercase': {},
        'change_case:english_capital': {},
        'punctuation:no_comma': {},
    }
    FORBID_POOL = (
        'however', 'important', 'example', 'different',
        'consider', 'information', 'specific', 'provide',
    )

    def kw_for(base_key, typ):
        if typ != 'keywords:forbidden_words':
            return dict(TYPE_TO_KW[typ])
        rng = np.random.default_rng(SEED + int(base_key) * 17)
        return {'forbidden_words': sorted(map(str, rng.choice(FORBID_POOL, 2, replace=False)))}

    def wording(typ, kw, form):
        assert form in FORMS
        if typ == 'keywords:forbidden_words':
            words = ', '.join(json.dumps(w, ensure_ascii=False) for w in kw['forbidden_words'])
            opts = (
                f'Do not use any of these words in your response: {words}.',
                f'Never include these words in your answer: {words}.',
                f'Avoid every word listed here in your reply: {words}.',
                f'Excluded vocabulary: none of {words} may appear in your response.',
                f'Ensure your answer contains zero occurrences of these words: {words}.',
            )
        elif typ == 'detectable_content:number_placeholders':
            n = kw['num_placeholders']
            opts = (
                f'Include at least {n} square-bracket placeholders such as [name] in your answer.',
                f'Your response must contain no fewer than {n} [name]-style placeholders.',
                f'Add {n} or more placeholders in square brackets, for example [name].',
                f'Placeholder requirement: use at least {n} items formatted like [name].',
                f'Provide a minimum of {n} square-bracketed placeholders (e.g., [name]) in your reply.',
            )
        elif typ == 'startend:end_checker':
            end = json.dumps(kw['end_phrase'], ensure_ascii=False)
            opts = (
                f'End your response with the exact phrase {end}.',
                f'The final words of your answer must be {end}.',
                f'Required ending: finish with {end}, and write nothing afterward.',
                f'Your reply must conclude with this exact phrase: {end}.',
                f'Make {end} the last phrase in your response.',
            )
        elif typ == 'change_case:english_lowercase':
            
            opts = (
                'Write your entire response in English using only lowercase letters.',
                'Answer in English, and do not use any uppercase letters.',
                'Use English throughout your reply; every letter must be lowercase.',
                'All text in your answer must be English and written in lowercase.',
                'Your response must be in English with no capital letters anywhere.',
            )
        elif typ == 'change_case:english_capital':
            
            opts = (
                'Write your entire response in English using only uppercase letters.',
                'Answer in English, and do not use any lowercase letters.',
                'Use English throughout your reply; every letter must be uppercase.',
                'All text in your answer must be English and written in capital letters.',
                'Your response must be in English with no small letters anywhere.',
            )
        elif typ == 'punctuation:no_comma':
            opts = (
                'Do not use commas anywhere in your response.',
                'Your answer must contain no comma punctuation marks.',
                'Avoid commas throughout your reply.',
                'Use zero commas in your entire response.',
                'Never put a comma in your answer.',
            )
        else:
            raise ValueError(f'Unknown instruction type: {typ}')
        assert len(opts) == len(FORMS) and len(set(opts)) == len(FORMS)
        return opts[FORMS.index(form)]

    def compatible(base, typ, kw):
        
        
        text = base['prompt'].lower()
        if not 45 <= len(text) <= 4800:
            return False
        if typ.startswith('change_case:') and re.search(
            r'(?i)json|xml|csv|case-sensitive|capitaliz|all-caps|all caps|'
            r'uppercase|lowercase|case of', text
        ):
            return False
        if typ == 'punctuation:no_comma' and re.search(
            r'(?i)csv|comma.separated|commas? between|separated by commas?', text
        ):
            return False
        if typ == 'startend:end_checker' and re.search(
            r'(?i)end (?:your |the )?(?:response|answer|reply) with|'
            r'finish (?:your |the )?(?:response|answer|reply) with|'
            r'last line must|only (?:json|xml)', text
        ):
            return False
        if typ == 'detectable_content:number_placeholders' and re.search(
            r'(?i)no square brackets|do not use brackets|only (?:one|a single) word|'
            r'single word answer|only (?:json|xml)', text
        ):
            return False
        if typ == 'keywords:forbidden_words' and any(
            re.search(r'(?i)\b' + re.escape(w) + r'\b', text)
            for w in kw['forbidden_words']
        ):
            return False
        return True

    def overlay(base_key, typ, form):
        base_key = int(base_key)
        base = original_by_key[base_key]
        kw = kw_for(base_key, typ)
        assert compatible(base, typ, kw), (base_key, typ)
        prompt = (
            base['prompt'].rstrip() +
            '\n\nAdditional requirement for your answer:\n' +
            wording(typ, kw, form)
        )
        return {
            'key': base_key, 'type': typ, 'variant': form,
            'prompt': prompt, 'target_kwargs': kw,
            'category': base['category'],
            'native_level': base['level'],
            'family': base['family'],
        }

    
    
    positive_text = {
        'keywords:forbidden_words': 'The children walked through the garden to read a book.',
        'detectable_content:number_placeholders': 'Use [name], [location], and [date] in the document.',
        'startend:end_checker': 'The explanation is complete. This concludes my response.',
        'change_case:english_lowercase': (
            'this is an english sentence. every word uses lowercase letters. '
            'the children are reading a book in a quiet garden.'
        ),
        'change_case:english_capital': (
            'THIS IS AN ENGLISH SENTENCE. EVERY LETTER IS CAPITALIZED. '
            'THE CHILDREN ARE READING A BOOK IN A QUIET GARDEN.'
        ),
        'punctuation:no_comma': 'This is a complete English sentence without commas.',
    }

    
    
    def negative_text(typ, kw):
        if typ == 'keywords:forbidden_words':
            return 'The word ' + kw['forbidden_words'][0] + ' appears here.'
        if typ == 'detectable_content:number_placeholders':
            return 'The example contains no square bracket placeholders.'
        if typ == 'startend:end_checker':
            return 'The response does not end with the required closing phrase.'
        if typ == 'change_case:english_lowercase':
            return 'THIS RESPONSE IS WRITTEN IN CAPITAL ENGLISH LETTERS.'
        if typ == 'change_case:english_capital':
            return 'this response is written in lowercase english letters.'
        if typ == 'punctuation:no_comma':
            return 'This answer has one comma, which must fail.'
        raise ValueError(typ)

    smoke = []
    for typ in CANDIDATE_TYPES:
        k = next(
            (int(r['key']) for r in raw if compatible(r, typ, kw_for(r['key'], typ))),
            None,
        )
        assert k is not None, f'No compatible FollowBench source prompt for {typ}'
        kw = kw_for(k, typ)
        pos = positive_text[typ]
        neg = negative_text(typ, kw)
        records = [overlay(k, typ, v) for v in FORMS]
        assert len({r['prompt'] for r in records}) == len(FORMS)
        assert all(r['target_kwargs'] == kw for r in records)
        assert all(r['prompt'].startswith(original_by_key[k]['prompt'].rstrip()) for r in records)
        for r in records:
            p = int(score_one(typ, kw, pos, r['prompt']))
            n = int(score_one(typ, kw, neg, r['prompt']))
            assert p == 1 and n == 0, (
                f'Verifier mismatch: type={typ}, form={r["variant"]}, '
                f'positive={p}, negative={n}; inspect checker semantics.'
            )
            smoke.append((typ, r['variant'], p, n))

    assert len(smoke) == len(CANDIDATE_TYPES) * len(FORMS)
    print(f'PASS: {len(CANDIDATE_TYPES)} types x {len(FORMS)} formulations; '
          f'{2 * len(smoke)} positive/negative official-checker tests.')
    print('FollowBench source prompts unchanged; native FollowBench constraints are not graded.')

In [ ]:
if not REPLAY_COMPLETE:
    
    
    texts=[r['prompt'] for r in raw]
    TF=TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=20000,min_df=1).fit_transform(texts)
    sim=cosine_similarity(TF).astype(np.float32)
    parent=list(range(len(raw)))
    def root(i):
        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
        return i
    def union(i,j):
        a,b=root(i),root(j)
        if a!=b:parent[max(a,b)]=min(a,b)
    for i in range(len(raw)):
        for j in range(i+1,len(raw)):
            if sim[i,j]>=.91:union(i,j)
    
    first={}
    for i,r in enumerate(raw):
        if r['family'] in first:union(i,first[r['family']])
        else:first[r['family']]=i
    clusters=defaultdict(list)
    for i,r in enumerate(raw):clusters[root(i)].append(int(r['key']))
    GROUP={k:min(ks) for ks in clusters.values() for k in ks}
    rep_by_group={min(ks):min(ks) for ks in clusters.values()}
    keys_by_group={min(ks):sorted(ks,key=lambda k:original_by_key[k]['level']) for ks in clusters.values()}
    coverage=[]
    for g,k in sorted(rep_by_group.items()):
        base=original_by_key[k]
        rec={'group':g,'key':k,'category':base['category'],'native_level':base['level'],
             'source_level_count':len(keys_by_group[g])}
        for t in CANDIDATE_TYPES:
            rec[t]=int(any(compatible(original_by_key[kk],t,kw_for(kk,t)) for kk in keys_by_group[g]))
        coverage.append(rec)
    coverage=pd.DataFrame(coverage)
    coverage.to_csv(DATA/'preflight_type_coverage.csv',index=False)

    
    
    usable=coverage[coverage[list(CANDIDATE_TYPES)].sum(axis=1)>=3]
    assert len(usable)>=N_PILOT_GROUPS+MIN_STUDY_GROUPS,(
        f'Only {len(usable)} independent compatible families; need at least '
        f'{N_PILOT_GROUPS+MIN_STUDY_GROUPS}. Do not run GPU generation.')
    rng=np.random.default_rng(SEED+8)
    chosen=rng.permutation(usable.group.to_numpy(int))
    PILOT_GROUPS=sorted(map(int,chosen[:N_PILOT_GROUPS]))
    STUDY_GROUPS=sorted(map(int,chosen[N_PILOT_GROUPS:N_PILOT_GROUPS+MAX_STUDY_GROUPS]))
    assert len(STUDY_GROUPS)>=MIN_STUDY_GROUPS
    assert not (set(PILOT_GROUPS)&set(STUDY_GROUPS))
    sub=coverage[coverage.group.isin(STUDY_GROUPS)]
    counts={t:int(sub[t].sum()) for t in CANDIDATE_TYPES}
    assert sum(n>=MIN_STUDY_GROUPS-15 for n in counts.values())>=3,(
        f'Fewer than three constraint families have broad source coverage: {counts}')
    freeze(DATA/'task_groups.json',{'near_duplicate_threshold':.91,
      'pilot_groups':PILOT_GROUPS,'study_groups':STUDY_GROUPS,
      'group_map':{str(k):int(v) for k,v in GROUP.items()}})
    proto={'study':'followbench_derived_controlled_v1','source_sha256':{c:shafile(p) for c,p in fb_source_files.items()},
     'model':MODEL_ID,'revision':MODEL_REV,'verifier_sha':verifier_sha,
     'budget':GEN_BUDGET,'max_input_tokens':MAX_INPUT_TOKENS,'layers':list(LAYERS),
     'folds':FOLDS,'seed':SEED,'pilot_groups':PILOT_GROUPS,'study_groups':STUDY_GROUPS,
     'C':list(Cs),'delta_penalties':list(DELTA_PENALTIES),'margin':ID_MARGIN,
     'methods':list(METHODS),'formulations':['original','P1','P2','R1','R2'],
     'label':'added deterministic target constraint ONLY; NOT official FollowBench native constraint satisfaction'}
    freeze(ROOT/'protocol.json',proto)
    print('Preflight PASS. FollowBench evolution/near-duplicate groups:',len(clusters))
    print('Pilot:',len(PILOT_GROUPS),'| study:',len(STUDY_GROUPS),'| compatible task families:',counts)
    print('Study source-level prompts:',sum(len(keys_by_group[g]) for g in STUDY_GROUPS))
    
    assert len(set(GROUP.values()))==len(clusters)

### 2.2 Model inference, protocol screening, and training representations

In [ ]:
if not REPLAY_COMPLETE:
    GEN_CACHE=CACHE/'generations'; STATE_CACHE=CACHE/'states'
    GEN_CACHE.mkdir(exist_ok=True);STATE_CACHE.mkdir(exist_ok=True)
    MODEL_CONTEXT={'id':MODEL_ID,'revision':MODEL_REV,'transformers':'4.53.3',
      'chat_template':'single_user_add_generation_prompt','sampling':'greedy','max_input':MAX_INPUT_TOKENS,
      'attention':'native_sdpa','weight_dtype':('bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16')}
    model=None;tokenizer=None;INPUT_DEVICE=None

    def ensure_model():
        global model,tokenizer,INPUT_DEVICE
        if model is not None:return
        assert torch.cuda.is_available(), 'Select a GPU runtime for generation.'
        import transformers
        assert transformers.__version__=='4.53.3',f'Expected Transformers 4.53.3, found {transformers.__version__}'
        from transformers import AutoTokenizer,AutoModelForCausalLM
        tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REV,trust_remote_code=False)
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REV,
             torch_dtype=dtype,device_map='auto',trust_remote_code=False,attn_implementation='sdpa')
        model.eval();INPUT_DEVICE=model.get_input_embeddings().weight.device
        assert INPUT_DEVICE.type=='cuda' and model.config.hidden_size==3072 and len(model.model.layers)==32
        assert model.__class__.__module__.startswith('transformers.models.phi3.')
        batch,n=enc_prompt('Say hello.')
        with torch.inference_mode():
            output=model.generate(**batch,max_new_tokens=3,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id)
            assert output.shape[-1]>n
        del output,batch
        print('Phi-3 native SDPA smoke test passed.')

    def enc_prompt(prompt):
        encoded=tokenizer.apply_chat_template([{'role':'user','content':prompt}],
           tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors='pt')
        n=int(encoded['input_ids'].shape[-1])
        if n>MAX_INPUT_TOKENS:raise ValueError(f'Input {n} tokens exceeds MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}')
        return {k:v.to(INPUT_DEVICE) for k,v in encoded.items()},n

    def gen_args(r):
        return {'prompt_sha':sha(r['prompt'].encode()),'budget':GEN_BUDGET,'context':MODEL_CONTEXT,
                'verifier':verifier_sha,'type':r.get('type'),'kwargs':r.get('target_kwargs'),
                'category':r.get('category'),'native_level':r.get('native_level')}
    def state_args(r):
        return {'prompt_sha':sha(r['prompt'].encode()),'context':MODEL_CONTEXT,
                'layers':list(LAYERS),'readout':'final_prompt_token'}

    def cache_gen(r):
        args=gen_args(r);p=GEN_CACHE/(sha(canon(args).encode())+'.json')
        if p.exists():
            item=json.loads(p.read_text());assert item['fingerprint']==args
            flags=score_record(r,item['response'])
            assert flags==item['scoring'], 'Scored cache does not match official verifier.'
            return item,True
        return None,False

    def score_record(r,response):
        target=score_one(r['type'],r['target_kwargs'],response,r['prompt'])
        return {'target':target}

    def gen_run(r):
        ensure_model();enc,n=enc_prompt(r['prompt'])
        with torch.inference_mode():
            out=model.generate(**enc,max_new_tokens=GEN_BUDGET,do_sample=False,
                  use_cache=True,pad_token_id=tokenizer.eos_token_id)
        seq=out[0,n:];nout=int(seq.numel())
        ends={int(tokenizer.eos_token_id)}
        ends.update(map(int,getattr(tokenizer,'additional_special_tokens_ids',[]) or []))
        cap=bool(nout>=GEN_BUDGET and (not nout or int(seq[-1]) not in ends))
        response=tokenizer.decode(seq,skip_special_tokens=True).strip()
        del seq,enc,out
        return {'response':response,'output_tokens':nout,'input_tokens':n,'capped':cap,
                'scoring':score_record(r,response),'fingerprint':gen_args(r)}

    def generation(r):
        item,hit=cache_gen(r)
        if item is not None:return item,hit
        item=gen_run(r)
        save_json(GEN_CACHE/(sha(canon(item['fingerprint']).encode())+'.json'),item)
        return item,False

    def state(r):
        args=state_args(r);stem=sha(canon(args).encode());p=STATE_CACHE/(stem+'.npz');meta=STATE_CACHE/(stem+'.json')
        if p.exists() and meta.exists():
            assert json.loads(meta.read_text())==args
            with np.load(p,allow_pickle=False) as f:z=np.asarray(f['states'],np.float32)
            assert z.shape==(len(LAYERS),3072) and np.isfinite(z).all()
            return z,True
        ensure_model();enc,n=enc_prompt(r['prompt'])
        with torch.inference_mode():
            out=model(**enc,output_hidden_states=True,use_cache=False,return_dict=True)
            z=np.stack([out.hidden_states[j][0,-1].float().cpu().numpy() for j in LAYERS]).astype(np.float32)
            del out,enc
        assert z.shape==(len(LAYERS),3072) and np.isfinite(z).all()
        save_npz(p,states=z.astype(np.float16));save_json(meta,args)
        return z,False

    def result(r,with_state=False):
        item,hit=generation(r)
        return (item,state(r)[0],hit) if with_state else (item,None,hit)
    print('Cache ready. Old generations are never reused under a different token budget.')

In [ ]:
if not REPLAY_COMPLETE:
    pilot_rows=[]
    for group,key in tqdm([(g,k) for g in PILOT_GROUPS for k in keys_by_group[g]],desc='Pilot (original/P1 only)'):
        for typ in CANDIDATE_TYPES:
            kw=kw_for(key,typ)
            if not compatible(original_by_key[key],typ,kw):continue
            for form in ('original','P1'):
                r=overlay(key,typ,form);it,_,_=result(r)
                pilot_rows.append({'group':group,'key':key,'type':typ,'variant':form,
                    'target':it['scoring']['target'],'native_level':r['native_level'],'category':r['category'],
                    'capped':int(it['capped'])})
    pilot=pd.DataFrame(pilot_rows)
    assert len(pilot)>0
    pilot.to_csv(RESULTS/'pilot.csv',index=False)
    summary=[]
    for typ,frame in pilot.groupby('type'):
        n=len(frame);success=int(frame.target.sum());failure=n-success;cap=float(frame.capped.mean())
        pos_groups=frame[frame.target.eq(1)].group.nunique()
        neg_groups=frame[frame.target.eq(0)].group.nunique()
        summary.append({'type':typ,'n':n,'success':success,'failure':failure,'cap_rate':cap,
           'groups':frame.group.nunique(),'positive_groups':pos_groups,'negative_groups':neg_groups,
           'eligible':bool(min(success,failure)>=PILOT_MIN_CLASS and
             min(pos_groups,neg_groups)>=4 and cap<=PILOT_MAX_CAP)})
    screen=pd.DataFrame(summary).sort_values('type');screen.to_csv(RESULTS/'pilot_screen.csv',index=False)
    display(screen.assign(cap_pct=100*screen.cap_rate).drop(columns=['cap_rate']).round(2))
    eligible=screen[screen.eligible].copy()
    eligible=eligible[eligible.type.map(counts)>=MIN_STUDY_GROUPS-15].copy()
    if len(eligible)<PRIMARY_N_TYPES:
        raise RuntimeError('Fewer than three balanced, low-truncation types. This is a DATA limitation. '
         'Inspect pilot_screen.csv and test new predeclared constraint categories in a NEW study version.')
    eligible['balance']=np.minimum(eligible.success,eligible.failure)/eligible.n
    eligible=eligible.sort_values(['balance','groups','type'],ascending=[False,False,True])
    PRIMARY_TYPES=eligible.type.tolist()[:PRIMARY_N_TYPES]
    assert len(PRIMARY_TYPES)>=2
    TYPE_TO_ID={t:i for i,t in enumerate(PRIMARY_TYPES)};N_TYPES=len(PRIMARY_TYPES)
    freeze(ROOT/'selected_types.json',{'types':PRIMARY_TYPES,'pilot_sha':shafile(RESULTS/'pilot.csv'),
        'selection':'largest minimum class proportion among pilot-eligible types; no R1/R2 outcome used'})
    print('Selected primary types:',PRIMARY_TYPES)
    print('Study groups remain fully held out:',len(STUDY_GROUPS))

In [ ]:
if not REPLAY_COMPLETE:
    train_rows=[];train_z=[]
    for group,key in tqdm([(g,k) for g in STUDY_GROUPS for k in keys_by_group[g]],desc='Training formulation caches'):
        for typ in PRIMARY_TYPES:
            kw=kw_for(key,typ)
            if not compatible(original_by_key[key],typ,kw):continue
            for form in ('original','P1','P2'):
                r=overlay(key,typ,form)
                item,z,_=result(r,with_state=True)
                train_rows.append({'group':group,'key':key,'type':typ,'variant':form,
                  'prompt':r['prompt'],'label':item['scoring']['target'],
                  'category':r['category'],'native_level':r['native_level'],'capped':int(item['capped']),
                  'tokens':int(item['output_tokens'])})
                train_z.append(z)
    TR=pd.DataFrame(train_rows);Ztrain=np.stack(train_z).astype(np.float32)
    assert len(TR)==len(Ztrain) and not TR.duplicated(['key','type','variant']).any()
    assert set(TR.variant)=={'original','P1','P2'}
    TR.to_json(RESULTS/'training_1536.jsonl',orient='records',lines=True,force_ascii=False)
    np.save(RESULTS/'training_states.npy',Ztrain.astype(np.float16))
    cap=TR.groupby(['type','variant']).agg(n=('label','size'),success=('label','sum'),capped=('capped','mean'))
    cap['fail']=cap.n-cap.success;cap.to_csv(RESULTS/'training_label_support.csv');display(cap.assign(capped_pct=100*cap.capped).drop(columns=['capped']).round(2))
    if (cap.capped>.10).any():print('WARNING: high cap rate in some training cells. Include it in limitations; consider a NEW higher-budget protocol.')
    GROUPS=np.array(sorted(TR.group.unique()),dtype=int)
    assert len(GROUPS)>=FOLDS*10, 'Too few independent groups for five-fold evaluation.'
    print('Training examples:',len(TR),'| independent groups:',len(GROUPS))

### 2.3 Four readouts and held-out formulations

In [ ]:
if not REPLAY_COMPLETE:
    from scipy.special import expit
    METHODS=('B0','B2','MT0','MT2')
    N_TYPES=len(PRIMARY_TYPES)
    TR=TR.reset_index(drop=True).copy()
    X=Ztrain[:,LAYERS.index(PRIMARY_LAYER)].astype(np.float32)
    assert len(TR)==len(X)
    GROUPS=np.array(sorted(set(TR.group)))
    outer=list(KFold(n_splits=FOLDS,shuffle=True,random_state=SEED).split(GROUPS))
    fold_for_group={int(GROUPS[i]):fold for fold,(_,te) in enumerate(outer) for i in te}
    assert len(fold_for_group)==len(GROUPS)
    for tr,te in outer:assert set(GROUPS[tr]).isdisjoint(GROUPS[te])

    class Probe:
        def __init__(self,mean,scale,w,delta,offset,b):
            self.mean=np.asarray(mean,np.float32);self.scale=np.asarray(scale,np.float32)
            self.w=np.asarray(w,np.float32);self.delta=np.asarray(delta,np.float32)
            self.offset=np.asarray(offset,np.float32);self.b=float(b)
        def predict(self,x,t):
            x=np.asarray(x,np.float32)
            if x.ndim==1:x=x[None,:]
            t=np.asarray(t,int).reshape(-1)
            assert len(x)==len(t) and x.shape[1]==len(self.w)
            z=(x-self.mean)/self.scale
            logits=np.sum(z*(self.w[None,:]+self.delta[t]),axis=1)+self.b+self.offset[t]
            return expit(np.clip(logits,-35,35))
        def save(self,path):
            tmp=path.with_suffix('.tmp')
            with open(tmp,'wb') as f:np.savez_compressed(f,mean=self.mean,scale=self.scale,
               w=self.w,delta=self.delta,offset=self.offset,b=np.array(self.b))
            os.replace(tmp,path)
        @classmethod
        def load(cls,path):
            with np.load(path,allow_pickle=False) as q:
                return cls(q['mean'],q['scale'],q['w'],q['delta'],q['offset'],float(q['b']))

    def fit_probe(train_groups,method,C,penalty=16.,layer=PRIMARY_LAYER,max_iter=90):
        tr=TR.group.isin(train_groups) & (TR.variant.eq('original') if method in ('B0','MT0') else True)
        idx=np.flatnonzero(tr.to_numpy());ixorig=np.flatnonzero((TR.group.isin(train_groups)&TR.variant.eq('original')).to_numpy())
        assert len(idx)>0 and len(ixorig)>0
        sc=StandardScaler().fit(X[ixorig] if layer==PRIMARY_LAYER else Ztrain[ixorig,LAYERS.index(layer)].astype(np.float32))
        XX=X[idx] if layer==PRIMARY_LAYER else Ztrain[idx,LAYERS.index(layer)].astype(np.float32)
        z=((XX-sc.mean_)/np.maximum(sc.scale_,1e-6)).astype(np.float32)
        y=TR.iloc[idx].label.to_numpy(np.float32)
        tt=TR.iloc[idx].type.map(TYPE_TO_ID).to_numpy(np.int64)
        if len(np.unique(y))<2:
            prior=np.clip(float(y.mean()),1e-5,1-1e-5)
            return Probe(sc.mean_,np.maximum(sc.scale_,1e-6),np.zeros(X.shape[1]),
                np.zeros((N_TYPES,X.shape[1])),np.zeros(N_TYPES),np.log(prior/(1-prior)))
        
        g=TR.iloc[idx].groupby(['group','type']).size().to_dict()
        wt=np.array([1/g[(int(gg),str(tt))] for gg,tt in TR.iloc[idx][['group','type']].itertuples(index=False,name=None)],np.float32);wt/=wt.sum()
        device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        xx=torch.as_tensor(z,device=device); yy=torch.as_tensor(y,device=device)
        ids=torch.as_tensor(tt,device=device);wts=torch.as_tensor(wt,device=device)
        w=torch.nn.Parameter(torch.zeros(X.shape[1],device=device))
        b=torch.nn.Parameter(torch.zeros((),device=device))
        mt=method.startswith('MT')
        delta=torch.nn.Parameter(torch.zeros((N_TYPES,3072),device=device)) if mt else None
        offsets=torch.nn.Parameter(torch.zeros(N_TYPES,device=device)) if mt else None
        params=[w,b]+([delta,offsets] if mt else [])
        opt=torch.optim.LBFGS(params,lr=1.,max_iter=max_iter,line_search_fn='strong_wolfe')
        ridge=1.0/(2*float(C)*int(len(g)))
        def closure():
            opt.zero_grad()
            v=torch.sum(xx*(w[None,:]+delta[ids] if mt else w[None,:]),dim=1)+b
            if mt:v=v+offsets[ids]
            loss=(wts*torch.nn.functional.binary_cross_entropy_with_logits(v,yy,reduction='none')).sum()+ridge*w.square().sum()
            if mt:loss=loss+ridge*float(penalty)*delta.square().sum()+ridge*offsets.square().sum()
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite logistic objective')
            loss.backward();return loss
        opt.step(closure)
        pp=Probe(sc.mean_,np.maximum(sc.scale_,1e-6),w.detach().cpu().numpy(),
            delta.detach().cpu().numpy() if mt else np.zeros((N_TYPES,X.shape[1])),
            offsets.detach().cpu().numpy() if mt else np.zeros(N_TYPES),b.detach().cpu().item())
        del xx,yy,ids,wts,opt
        return pp

    def macro_validation(probe,groups,variants,layer=PRIMARY_LAYER):
        mask=(TR.group.isin(groups)&TR.variant.isin(variants)).to_numpy();idx=np.flatnonzero(mask)
        if not len(idx):return np.nan
        XX=X if layer==PRIMARY_LAYER else Ztrain[:,LAYERS.index(layer)].astype(np.float32)
        ss=[]
        for typ in PRIMARY_TYPES:
            for v in variants:
                sub=idx[(TR.iloc[idx].type.eq(typ)&TR.iloc[idx].variant.eq(v)).to_numpy()]
                if not len(sub):continue
                yy=TR.iloc[sub].label.to_numpy(int)
                if len(np.unique(yy))<2:continue
                pp=probe.predict(XX[sub],np.full(len(sub),TYPE_TO_ID[typ]))
                ss.append(auc(yy,pp))
        return float(np.nanmean(ss)) if ss else np.nan

    FROZEN_FP=sha(canon({'dataset':proto['source_sha256'],'protocol':proto,
     'selected_types':PRIMARY_TYPES,'selected_keys':sorted(set(TR.key)),
     'label_file_sha':shafile(RESULTS/'training_1536.jsonl'),
     'layers':LAYERS,'methods':METHODS,'software':'torch_LBFGS_v3'}).encode())
    freeze(MODELS/'probe_model_fingerprint.json',{'sha256':FROZEN_FP})
    fitted={};training_log=[]
    for fold,(trix,teix) in enumerate(outer):
        trainval=GROUPS[trix];test=GROUPS[teix]
        inner_tr,inner_val=train_test_split(trainval,test_size=.25,random_state=SEED+fold)
        base_id=None
        for method in METHODS:
            path=MODELS/f'{fold}_{method}.npz';meta=MODELS/f'{fold}_{method}.json'
            if path.exists() and meta.exists():
                info=json.loads(meta.read_text());assert info['fp']==FROZEN_FP
                pp=Probe.load(path);fitted[(fold,method)]=pp;training_log.append(info)
                if method=='B0':base_id=info['id_auc']
                continue
            choices=[(c,16.) for c in Cs] if not method.startswith('MT') else list(itertools.product(Cs,DELTA_PENALTIES))
            scored=[]
            for c,p in choices:
                tmp=fit_probe(inner_tr,method,c,p)
                shift=macro_validation(tmp,inner_val,['P1','P2'])
                original=macro_validation(tmp,inner_val,['original'])
                if np.isfinite(original) and np.isfinite(shift):scored.append((shift,original,c,p))
            if not scored:
                print('Warning: validation has too few within-type class pairs; using prior-fixed regularization for',fold,method)
                scored=[(0.5,0.5,Cs[1],16.)]
            feasible=[z for z in scored if base_id is None or z[1]>=base_id-ID_MARGIN]
            best=max(feasible or scored,key=lambda z:(z[0],z[1],-z[2],-z[3]))
            pp=fit_probe(trainval,method,best[2],best[3],max_iter=110)
            pp.save(path)
            info={'fold':fold,'method':method,'fp':FROZEN_FP,'C':best[2],
                  'delta_penalty':best[3],'validation_shift_auc':best[0],'id_auc':best[1],
                  'constraint_feasible':bool(feasible),'train_groups':list(map(int,trainval)),
                  'heldout_groups':list(map(int,test))}
            save_json(meta,info);fitted[(fold,method)]=pp;training_log.append(info)
            if method=='B0':base_id=best[1]
    pd.DataFrame(training_log).drop(columns=['train_groups','heldout_groups']).to_csv(RESULTS/'probe_tuning.csv',index=False)
    assert len(fitted)==FOLDS*len(METHODS)
    print('All fitted probes:',len(fitted),' | frozen before final R1/R2 scoring')

In [ ]:
if not REPLAY_COMPLETE:
    
    test_rows=[];test_z=[]
    for group,key in tqdm([(int(g),k) for g in GROUPS for k in keys_by_group[int(g)]],desc='Final unseen R1/R2'):
        for typ in PRIMARY_TYPES:
            if not compatible(original_by_key[key],typ,kw_for(key,typ)):continue
            for form in ('R1','R2'):
                r=overlay(key,typ,form)
                item,z,_=result(r,with_state=True)
                test_rows.append({'group':int(group),'key':key,'type':typ,'variant':form,
                     'prompt':r['prompt'],'label':item['scoring']['target'],
                     'category':r['category'],'native_level':r['native_level'],'capped':int(item['capped']),
                     'tokens':int(item['output_tokens'])})
                test_z.append(z)
    TE=pd.DataFrame(test_rows);Ztest=np.stack(test_z).astype(np.float32)
    assert len(TE)==len(Ztest) and not TE.duplicated(['key','type','variant']).any()
    TE.to_json(RESULTS/'unseen_R_scored.jsonl',orient='records',lines=True,force_ascii=False)
    np.save(RESULTS/'unseen_R_states.npy',Ztest.astype(np.float16))
    print('Held-out test predictions:',len(TE),'| original independent groups:',TE.group.nunique())

### 2.4 Transfer, paired changes, and validity checks

In [ ]:
if not REPLAY_COMPLETE:
    
    PRED=[]
    for domain,frame,Z in [('ID',TR,Ztrain),('OOD',TE,Ztest)]:
        for ix,row in enumerate(frame.itertuples(index=False)):
            if domain=='ID' and row.variant!='original':continue
            fold=fold_for_group[int(row.group)]
            layer=Z[ix,LAYERS.index(PRIMARY_LAYER)]
            rec={'domain':domain,'group':int(row.group),'key':int(row.key),'type':row.type,
                 'variant':row.variant,'label':int(row.label),'category':row.category,'native_level':int(row.native_level),
                 'capped':int(row.capped),'tokens':int(row.tokens),'fold':fold,'prompt':row.prompt}
            for method in METHODS:
                rec[method]=float(fitted[(fold,method)].predict(layer,[TYPE_TO_ID[row.type]])[0])
            PRED.append(rec)
    P=pd.DataFrame(PRED);P.to_csv(RESULTS/'oof_predictions.csv',index=False)
    ID=P[P.domain.eq('ID')].copy();OOD=P[P.domain.eq('OOD')].copy()

    def macro_auc(frame,method):
        vals=[]
        target='label'
        for (_, _),s in frame.groupby(['type','variant']):
            v=auc(s[target],s[method])
            if np.isfinite(v):vals.append(v)
        return float(np.mean(vals)) if vals else np.nan
    summary=[]
    for domain,f in [('ID',ID),('OOD',OOD)]:
        for method in METHODS:
            summary.append({'domain':domain,'method':method,'macro_auc':macro_auc(f,method),
              'Brier':float(brier_score_loss(f.label,f[method])),
              'logloss':float(log_loss(f.label,np.clip(f[method],1e-7,1-1e-7),labels=[0,1]))})
    pd.DataFrame(summary).to_csv(RESULTS/'primary_metrics.csv',index=False)
    display(pd.DataFrame(summary).assign(AUROC=lambda d:100*d.macro_auc,Brier_x100=lambda d:100*d.Brier)[['domain','method','AUROC','Brier_x100']].round(2))

    cells=OOD.groupby(['type','variant']).agg(n=('label','size'),success=('label','sum'),capped=('capped','mean'))
    cells['failure']=cells.n-cells.success
    cells['auc_defined']=((cells.success>0)&(cells.failure>0))
    
    positive_groups=OOD[OOD.label.eq(1)].groupby(['type','variant']).group.nunique()
    negative_groups=OOD[OOD.label.eq(0)].groupby(['type','variant']).group.nunique()
    cells['positive_groups']=positive_groups.reindex(cells.index,fill_value=0)
    cells['negative_groups']=negative_groups.reindex(cells.index,fill_value=0)
    cells['adequate_class_support']=((cells.success>=MIN_TEST_CLASS_PER_FORM)&
     (cells.failure>=MIN_TEST_CLASS_PER_FORM)&
     (cells.positive_groups>=8)&(cells.negative_groups>=8))
    cells['acceptable_cap_rate']=cells.capped.le(PILOT_MAX_CAP+1e-12)
    cells.to_csv(RESULTS/'OOD_class_support.csv');display(cells.assign(cap_pct=100*cells.capped).drop(columns=['capped']).round(2))
    PRIMARY_EVALUABLE=bool(cells.adequate_class_support.all() and cells.acceptable_cap_rate.all())
    if not PRIMARY_EVALUABLE:
        print('CAUTION: held-out cells have insufficient independent binary class support or excess truncation. Do not claim confirmatory AUROC.')

    
    
    h1=[]
    for typ in PRIMARY_TYPES:
        o=ID[ID.type.eq(typ)][['key','group','label','capped']]
        for form in ('R1','R2'):
            q=OOD[(OOD.type.eq(typ))&(OOD.variant.eq(form))][['key','group','label','capped']]
            g=o.merge(q,on=['key','group'],suffixes=('_original','_new'),validate='one_to_one')
            if not len(g):continue
            a=g.label_original.to_numpy(int);b=g.label_new.to_numpy(int)
            up=int(((a==0)&(b==1)).sum());down=int(((a==1)&(b==0)).sum())
            means=(g.label_new-g.label_original).groupby(g.group).mean().to_numpy(float)
            observed=abs(float(means.mean()))
            rs=np.random.default_rng(SEED+len(h1)*127)
            nperm=20000
            signs=rs.choice([-1.,1.],size=(nperm,len(means)))
            null_abs=np.abs((signs*means[None,:]).mean(axis=1))
            p_group=float((1+(null_abs>=observed-1e-12).sum())/(nperm+1))
            h1.append({'type':typ,'form':form,'n':len(g),'independent_groups':g.group.nunique(),
              'success_original':float(a.mean()),'success_new':float(b.mean()),
              'delta':float((b-a).mean()),'failure_to_success':up,'success_to_failure':down,
              'group_randomization_p':p_group,'original_capped':float(g.capped_original.mean()),
              'new_capped':float(g.capped_new.mean()),'flip_rate':float((a!=b).mean())})
    h1=pd.DataFrame(h1)
    h1['holm_p']=holm(h1.group_randomization_p)
    h1.to_csv(RESULTS/'H1_paired.csv',index=False)
    h1_display=h1.copy()
    for col in ['success_original','success_new','delta','original_capped','new_capped','flip_rate']:
        h1_display[col+'_pct']=100*h1_display.pop(col)
    display(h1_display.round(2))

    
    def boot_delta(frame,method,baseline='B0',nboot=BOOTSTRAPS,seed=SEED):
        rg=np.random.default_rng(seed);groups=np.array(sorted(frame.group.unique()),int)
        loc={g:i for i,g in enumerate(groups)}
        series=[]
        for _,t in frame.groupby(['type','variant']):
            y=t.label.to_numpy(int);a=t[method].to_numpy(float);b=t[baseline].to_numpy(float)
            g=np.array([loc[int(k)] for k in t.group],int)
            series.append((y,a,b,g))
        values=[]
        for _ in range(nboot):
            cnt=rg.multinomial(len(groups),np.ones(len(groups))/len(groups))
            dif=[]
            for y,a,b,g in series:
                w=cnt[g]
                if w[y==0].sum()==0 or w[y==1].sum()==0:continue
                dif.append(roc_auc_score(y,a,sample_weight=w)-roc_auc_score(y,b,sample_weight=w))
            if dif:values.append(float(np.mean(dif)))
        values=np.asarray(values)
        return {'gain':macro_auc(frame,method)-macro_auc(frame,baseline),
                'ci_low':float(np.quantile(values,.025)) if len(values) else np.nan,
                'ci_high':float(np.quantile(values,.975)) if len(values) else np.nan,
                'one_sided_lower_95':float(np.quantile(values,.05)) if len(values) else np.nan,
                'valid_bootstraps':int(len(values))}
    contrasts=[('B2','B0'),('MT0','B0'),('MT2','B0'),('MT2','B2'),('MT2','MT0')]
    rows=[]
    for domain,f in [('OOD',OOD),('ID',ID)]:
        for j,(a,b) in enumerate(contrasts):
            rows.append({'domain':domain,'method':a,'baseline':b,
                        **boot_delta(f,a,b,seed=SEED+100*j+(domain=='ID'))})
    H2=pd.DataFrame(rows);H2['significant_95']=H2.ci_low.gt(0)
    H2['ID_noninferior_margin_002']=np.where(H2.domain.eq('ID'),H2.one_sided_lower_95>-ID_MARGIN,np.nan)
    H2.to_csv(RESULTS/'H2_bootstrap.csv',index=False)
    H2_display=H2.copy()
    for col in ['gain','ci_low','ci_high','one_sided_lower_95']:
        H2_display[col+'_points']=100*H2_display.pop(col)
    display(H2_display.round(2))

    sub=[]
    for domain,frame in [('ID',ID),('OOD',OOD)]:
        for (typ,form),g in frame.groupby(['type','variant']):
            for m in METHODS:
                sub.append({'domain':domain,'type':typ,'variant':form,'method':m,'n':len(g),
                  'positive':int(g.label.sum()),'negative':int(len(g)-g.label.sum()),
                  'auc':auc(g.label,g[m]),'Brier':float(brier_score_loss(g.label,g[m]))})
    pd.DataFrame(sub).to_csv(RESULTS/'per_type_formulation.csv',index=False)

    fold_rows=[]
    for (typ,form,fold),g in OOD.groupby(['type','variant','fold']):
        for m in METHODS:
            fold_rows.append({'type':typ,'form':form,'fold':int(fold),'method':m,'n':len(g),
                      'success':int(g.label.sum()),'auc':auc(g.label,g[m])})
    pd.DataFrame(fold_rows).to_csv(RESULTS/'within_fold_AUROC.csv',index=False)

    paired=[]
    for row in OOD.itertuples(index=False):
        old=ID[(ID.key==row.key)&(ID.type==row.type)]
        assert len(old)==1
        old=old.iloc[0];flip=int(row.label)-int(old.label)
        rr={'group':int(row.group),'key':int(row.key),'type':row.type,'form':row.variant,
            'behavior_change':flip,'changed':int(flip!=0)}
        for m in METHODS:
            delta=float(getattr(row,m)-old[m]);rr[m+'_score_shift']=delta
            rr[m+'_flip_correct']=int(delta*flip>0) if flip else np.nan
            rr[m+'_stable_delta']=abs(delta) if not flip else np.nan
        paired.append(rr)
    pd.DataFrame(paired).to_csv(RESULTS/'behavioral_flip_tracking.csv',index=False)
    print('Main H1/H2 analysis complete.')

In [ ]:
if not REPLAY_COMPLETE:
    control=OOD.copy()
    control['type_prior']=np.nan
    for i,row in control.iterrows():
        fold=int(row.fold);train_groups=set(GROUPS[outer[fold][0]])
        subset=TR[(TR.group.isin(train_groups))&(TR.type.eq(row.type))&(TR.variant.eq('original'))]
        control.at[i,'type_prior']=float((subset.label.sum()+1)/(len(subset)+2))
    if RUN_TEXT_CONTROL:
        control['char_ngram']=np.nan
        for fold,(itr,ite) in enumerate(outer):
            train_groups=GROUPS[itr];test_groups=GROUPS[ite]
            train=TR[(TR.group.isin(train_groups))&(TR.variant.eq('original'))]
            sub=control[control.group.isin(test_groups)]
            if len(sub)==0:continue
            vec=TfidfVectorizer(analyzer='char',ngram_range=(3,5),max_features=10000,sublinear_tf=True)
            a=vec.fit_transform(train.prompt);b=vec.transform(sub.prompt)
            if train.label.nunique()<2:probs=np.full(len(sub),float(train.label.mean()))
            else:
                clf=LogisticRegression(C=1.,solver='liblinear',max_iter=500)
                clf.fit(a,train.label);probs=clf.predict_proba(b)[:,1]
            control.loc[sub.index,'char_ngram']=probs
    assert control.type_prior.notna().all()
    if RUN_TEXT_CONTROL:assert control.char_ngram.notna().all()
    control.to_csv(RESULTS/'text_controls_predictions.csv',index=False)
    ms=list(METHODS)+['type_prior']+(['char_ngram'] if RUN_TEXT_CONTROL else [])
    ctrl=[{'model':m,'macro_auc':macro_auc(control,m),'Brier':float(brier_score_loss(control.label,control[m]))} for m in ms]
    pd.DataFrame(ctrl).to_csv(RESULTS/'text_controls_metrics.csv',index=False);display(pd.DataFrame(ctrl).assign(AUROC=lambda d:100*d.macro_auc,Brier_x100=lambda d:100*d.Brier)[['model','AUROC','Brier_x100']].round(2))

    if RUN_GEOMETRY:
        geom=[]
        for fold,(trix,teix) in enumerate(outer):
            gtr=GROUPS[trix]
            for typ in PRIMARY_TYPES:
                ix_type=TR.type.eq(typ)
                for form in ('P1','P2'):
                    
                    
                    vectors=[]
                    for v in ('original',form):
                        z=TR[ix_type&TR.variant.eq(v)&TR.group.isin(gtr)]
                        ix=z.index.to_numpy()
                        if z.label.nunique()<2:vectors=[];break
                        sc=StandardScaler().fit(Ztrain[ix,LAYERS.index(PRIMARY_LAYER)])
                        xx=sc.transform(Ztrain[ix,LAYERS.index(PRIMARY_LAYER)])
                        clf=LogisticRegression(C=.1,solver='liblinear',max_iter=1000)
                        clf.fit(xx,z.label.to_numpy(int))
                        
                        vdir=clf.coef_[0]/np.maximum(sc.scale_,1e-6)
                        vectors.append(vdir)
                    if len(vectors)==2:
                        a,b=vectors;cos=float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))
                        geom.append({'fold':fold,'type':typ,'form':form,'direction_cosine':cos})
        pd.DataFrame(geom).to_csv(RESULTS/'direction_geometry.csv',index=False)
        print('Direction comparisons:',len(geom))

In [ ]:
if not REPLAY_COMPLETE:
    
    source_cells=[]
    for (category,level,form),frame in OOD.groupby(['category','native_level','variant']):
        source_cells.append({'category':category,'native_level':int(level),'form':form,
            'n':len(frame),'success_pct':100*float(frame.label.mean()),
            'cap_pct':100*float(frame.capped.mean()),
            'B0_auc_pct':100*auc(frame.label,frame.B0),
            'MT2_auc_pct':100*auc(frame.label,frame.MT2)})
    pd.DataFrame(source_cells).to_csv(RESULTS/'followbench_source_category_level.csv',index=False)

    
    report=pd.read_csv(RESULTS/'primary_metrics.csv')
    report['AUROC_0_100']=100*report.macro_auc
    report['Brier_x100']=100*report.Brier
    display(report[['domain','method','AUROC_0_100','Brier_x100']].round(2))

    contrasts=pd.read_csv(RESULTS/'H2_bootstrap.csv')
    for k in ['gain','ci_low','ci_high','one_sided_lower_95']:
        contrasts[k+'_points']=100*contrasts[k]
    display(contrasts[contrasts.domain.eq('OOD')][
        ['method','baseline','gain_points','ci_low_points','ci_high_points']].round(2))

    manifest={'protocol':proto,'primary_label':'compliance with added deterministic IFEval-verifiable output constraint',
     'native_followbench_scoring':'NOT RUN. Native constraints often require LLM judging.',
     'n_native_rows':len(raw_entries),'n_native_families':len(base_families),
     'selected_types':PRIMARY_TYPES,'n_study_groups':len(GROUPS),
     'n_ood':len(OOD),'evaluable':PRIMARY_EVALUABLE,
     'study_warning':'Controlled FollowBench-derived extension, not native FollowBench benchmark HSR/SSR.',
     'bootstrap_unit':'source evolution family/near-duplicate cluster',
     'H1_p_values':'20k paired cluster sign-flip permutations, Holm adjusted',
     'primary_comparison':'MT2 vs B0 on heldout R1/R2 macro within-type AUROC'}
    save_json(ROOT/'RESULT_MANIFEST.json',manifest)
    archive=shutil.make_archive(str(ROOT/'FollowBench_Controlled_Results'),'zip',ROOT,'results')
    print('Results archive:',archive,'| All persistent caches:',ROOT)

In [ ]:
if not REPLAY_COMPLETE:
    
    import gc, torch
    if 'model' in globals() and model is not None:
        del model
        model = None
    if 'tokenizer' in globals() and tokenizer is not None:
        del tokenizer
        tokenizer = None
    gc.collect(); torch.cuda.empty_cache()
    FOLLOWBENCH_RESULTS = str(RESULTS)
    print('FollowBench-derived results:', FOLLOWBENCH_RESULTS)

# Unified final-paper results export

The source experiments retain their original statistical calculations and exports. The cells below consolidate **only** the planned paper outcomes into uniform tables and validity diagnostics, with one clear row per setting and method. **Macro-AUROC** is the mean of valid within-instruction-type × formulation cells, not a pooled AUROC. Paired-bootstrap uncertainty comes from each source pipeline's own task-grouped bootstrap; the unified table does not invent new confidence intervals.

If any source study aborts because verifier files changed, folds are invalid, or class support is missing, it is deliberately **not** silently skipped.

In [ ]:
if not REPLAY_COMPLETE:
    from pathlib import Path
    import pandas as pd, numpy as np, json
    from sklearn.metrics import roc_auc_score
    FINAL_ROOT = ULM_WORKDIR/'final_tables'
    FINAL_ROOT.mkdir(parents=True, exist_ok=True)
    METHOD_ORDER = ['B0','B2','MT0','MT2']

    
    simple_p = pd.read_csv(Path(SIMPLE_RESULTS)/'oof_original_and_R_predictions.csv')
    follow_p = pd.read_csv(Path(FOLLOWBENCH_RESULTS)/'oof_predictions.csv')

    def normalized(frame, setting):
        f=frame.copy()
        if setting=='IFEval-Simple':
            f=f.rename(columns={'task_id':'case'})
            f['family']=f['case']
        else:
            f=f.rename(columns={'group':'family','key':'case'})
        f['setting']=setting
        f['domain']=np.where(f['variant']=='original','original','held_out')
        assert f['variant'].isin(['original','R1','R2']).all()
        assert all(c in f.columns for c in ['setting','case','family','type','variant','label']+METHOD_ORDER)
        f['family']=f['family'].astype(str)
        f['case']=f['case'].astype(str)
        cols=['setting','case','family','type','variant','domain','label']+METHOD_ORDER
        for optional in ['fold','capped','output_tokens','type_only']:
            if optional in f.columns:cols.append(optional)
        return f[cols].copy()

    simple_p=simple_p[simple_p.variant.isin(['original','R1','R2'])].copy()
    follow_p=follow_p[follow_p.variant.isin(['original','R1','R2'])].copy()
    
    ALL=pd.concat([normalized(simple_p,'IFEval-Simple'),
                  normalized(follow_p,'FollowBench-derived')],ignore_index=True)
    assert ALL.setting.nunique()==2
    assert not ALL.duplicated(['setting','case','type','variant']).any(), 'Duplicate example/type/formulation; inspect source grouping.'
    assert ALL.groupby(['setting','case','type']).variant.nunique().eq(3).all(), 'Incomplete original/R1/R2 task pairs.'
    ALL.to_csv(FINAL_ROOT/'all_out_of_fold_predictions.csv',index=False)
    print('Verified out-of-fold predictions',len(ALL),'across',ALL.setting.nunique(),'settings')

    def safe_auc(y,score):
        y=np.asarray(y,int);p=np.asarray(score,float)
        return float(roc_auc_score(y,p)) if np.unique(y).size==2 else np.nan

    summary=[]
    within=[]
    for (setting,domain),frame in ALL.groupby(['setting','domain']):
        for method in METHOD_ORDER:
            cells=[]
            for (typ,variant),g in frame.groupby(['type','variant']):
                auc=safe_auc(g.label,g[method]);cells.append(auc)
                within.append({'setting':setting,'domain':domain,'type':typ,'formulation':variant,
                              'method':method,'n':len(g),'positive':int(g.label.sum()),
                              'negative':int(len(g)-g.label.sum()),'AUROC':auc})
            values=np.array(cells,float)
            summary.append({'setting':setting,'domain':domain,'method':method,
                            'n':int(len(frame)),'n_families':int(frame.family.nunique()),
                            'AUROC_macro':float(np.nanmean(values)),
                            'AUROC_pooled_descriptive':safe_auc(frame.label,frame[method]),
                            'valid_cells':int(np.isfinite(values).sum()),'total_cells':len(values)})
    summary=pd.DataFrame(summary);within=pd.DataFrame(within)
    summary.to_csv(FINAL_ROOT/'paper_table_main_AUROC.csv',index=False)
    within.to_csv(FINAL_ROOT/'paper_table_within_type_AUROC.csv',index=False)

    
    wide=summary.pivot(index='setting',columns=['domain','method'],values='AUROC_macro')
    changes=[]
    for setting in wide.index:
        changes.append({'setting':setting,
          'B0_original':100*wide.loc[setting,('original','B0')],
          'B0_heldout':100*wide.loc[setting,('held_out','B0')],
          'B0_transfer_delta_pp':100*(wide.loc[setting,('held_out','B0')]-wide.loc[setting,('original','B0')]),
          'B2_heldout':100*wide.loc[setting,('held_out','B2')],
          'MT0_heldout':100*wide.loc[setting,('held_out','MT0')],
          'MT2_heldout':100*wide.loc[setting,('held_out','MT2')],
          'MT2_minus_B2_pp':100*(wide.loc[setting,('held_out','MT2')]-wide.loc[setting,('held_out','B2')]),
          'MT2_minus_B0_original_pp':100*(wide.loc[setting,('original','MT2')]-wide.loc[setting,('original','B0')])})
    main_table=pd.DataFrame(changes)
    main_table.to_csv(FINAL_ROOT/'paper_table_main_comparisons.csv',index=False)
    display(main_table.round(2))

### Matched-pair diagnostic (not a causal attribution test)

A compliance flip is a change in the verified response label between two formulations of the same requirement. Rows are matched by **example ID** (not cluster/family ID). The score-change direction is evaluated **only among observed flips**, with support counts; cases with identical labels are tabulated separately. The score is a prediction of outcome, not a claim that the direction causes compliance.

In [ ]:
if not REPLAY_COMPLETE:
    from scipy.stats import binomtest
    pairs=[]
    for setting,df in ALL.groupby('setting'):
        anchor=df[df.variant.eq('original')].set_index(['case','type'])
        for form in ['R1','R2']:
            reform=df[df.variant.eq(form)].set_index(['case','type'])
            assert anchor.index.is_unique and reform.index.is_unique
            assert set(anchor.index)==set(reform.index)
            joined=anchor[['label']+METHOD_ORDER].join(reform[['label']+METHOD_ORDER],lsuffix='_O',rsuffix='_R',validate='one_to_one')
            for m in METHOD_ORDER:
                y0=joined.label_O.to_numpy(int);y1=joined.label_R.to_numpy(int)
                delta=y1-y0; ds=joined[f'{m}_R'].to_numpy(float)-joined[f'{m}_O'].to_numpy(float)
                flips=delta!=0; n=int(flips.sum());correct=int((ds[flips]*delta[flips]>0).sum())
                ties=int((ds[flips]==0).sum());denom=n 
                pairs.append({'setting':setting,'formulation':form,'method':m,
                  'matched_pairs':len(joined),'changed_pairs':n,'unchanged_pairs':int((~flips).sum()),
                  'failure_to_success':int((delta==1).sum()),'success_to_failure':int((delta==-1).sum()),
                  'flip_direction_correct':correct,'score_ties_on_flips':ties,
                  'flip_direction_accuracy':float(correct/denom) if denom else np.nan,
                  'flip_fraction':float(n/len(joined))})
    PAIR=pd.DataFrame(pairs)
    
    
    BOOT_N = 2000
    flip_uncertainty=[]
    for setting,df in ALL.groupby('setting'):
        families=sorted(df.family.unique())
        ix={fam:i for i,fam in enumerate(families)}
        rng=np.random.default_rng(20261017+len(setting))
        weights=rng.multinomial(len(families),np.full(len(families),1/len(families)),size=BOOT_N)
        a=df[df.variant.eq('original')].set_index(['case','type'])
        for form in ['R1','R2']:
            b=df[df.variant.eq(form)].set_index(['case','type'])
            z=a[['family','label']+METHOD_ORDER].join(b[['label']+METHOD_ORDER],rsuffix='_R',validate='one_to_one')
            dy=z.label_R.to_numpy(int)-z.label.to_numpy(int)
            valid=dy!=0
            fg=np.array([ix[fam] for fam in z.family],int)
            counts=np.bincount(fg,weights=valid.astype(float),minlength=len(families))
            n_flips=int(valid.sum());up=int((dy==1).sum());down=int((dy==-1).sum())
            for method in METHOD_ORDER:
                diff=z[f'{method}_R'].to_numpy(float)-z[method].to_numpy(float)
                success=(diff*dy>0)&valid
                correct_by_family=np.bincount(fg,weights=success.astype(float),minlength=len(families))
                numer=weights@correct_by_family; denom=weights@counts
                samples=np.divide(numer,denom,out=np.full(BOOT_N,np.nan),where=denom>0)
                good=samples[np.isfinite(samples)]
                flip_uncertainty.append({'setting':setting,'formulation':form,'method':method,
                    'n_flips':n_flips,'flip_direction_accuracy':float(success.sum()/n_flips) if n_flips else np.nan,
                    'ci_low':float(np.quantile(good,0.025)) if len(good) else np.nan,
                    'ci_high':float(np.quantile(good,0.975)) if len(good) else np.nan,
                    'n_valid_bootstraps':int(len(good)),
                    
                    'majority_flip_direction_oracle':float(max(up,down)/n_flips) if n_flips else np.nan})
    PAIR_CI=pd.DataFrame(flip_uncertainty)
    PAIR_CI.to_csv(FINAL_ROOT/'paper_table_paired_flip_uncertainty.csv',index=False)
    PAIR.to_csv(FINAL_ROOT/'paper_table_paired_flip_diagnostic.csv',index=False)
    display(PAIR[PAIR.method.isin(['B0','MT2'])][['setting','formulation','method','changed_pairs',
       'flip_direction_accuracy','flip_fraction']].round(3))

### Frozen-formulation checks

Automatic checker agreement does **not** establish semantic equivalence.

### Required provenance and result validity

Each source experiment already exports its training-fold uncertainty analysis. These checks verify that the unified report references those existing statistical files, including the requested original-domain retention tests, class-support audits, and independent text controls. 

In [ ]:
if not REPLAY_COMPLETE:
    import shutil, hashlib, zipfile, os
    required={
      'IFEval-Simple':{
        'dir':Path(SIMPLE_RESULTS),
        'files':['H2_paired_OOD_AUROC_bootstrap.csv','original_domain_retention.csv',
                 'flip_prediction_diagnostic.csv','char_tfidf_control.csv','per_type_formulation_metrics.csv']},
      'FollowBench-derived':{
        'dir':Path(FOLLOWBENCH_RESULTS),
        'files':['H2_bootstrap.csv','text_controls_metrics.csv','OOD_class_support.csv',
                 'behavioral_flip_tracking.csv','per_type_formulation.csv']}}
    for name,info in required.items():
        missing=[fname for fname in info['files'] if not (info['dir']/fname).is_file()]
        assert not missing, f'{name}: expected source statistical outputs are missing: {missing}'

    audit={
     'model':'microsoft/Phi-3-mini-128k-instruct',
     'frozen_revision':'f3c06aed622e14ca0abf5115094e4fc9a9948f36',
     'source_dirs':{'ifeval_simple':str(SIMPLE_RESULTS),'followbench_derived':str(FOLLOWBENCH_RESULTS)},
     'main_research_questions':['formulation_transfer','type_aware_vs_equally_augmented_shared',
                                'within_pair_predictive_reliability'],
     'protocol_cautions':['FollowBench-derived labels are for added verifiable rules',
     'Looped-model extensions, B3 and extra layer searches are excluded; CCT is a separate group-refit validation section'],
     'n_matched_rows':int(len(ALL)),
     'sha256':{}
    }
    for f in sorted(FINAL_ROOT.glob('*.csv')):
        audit['sha256'][f.name]=hashlib.sha256(f.read_bytes()).hexdigest()
    (FINAL_ROOT/'FINAL_PROTOCOL_AND_FILE_MANIFEST.json').write_text(json.dumps(audit,indent=2))
    archive=FINAL_ROOT/'Phi3_Final_Paper_Tables_and_Audits.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(FINAL_ROOT.glob('*')):
            if f.is_file() and f != archive:z.write(f,arcname=f.name)
        for name,info in required.items():
            for fname in info['files']:
                z.write(info['dir']/fname,arcname=f'source_audits/{name}/{fname}')
        
    print('Final paper files:',FINAL_ROOT)
    print('Portable evidence archive:',archive)


---

### Source lineage and honest reproducibility statement

**Original code bases incorporated:**
- `ULM_IFEvalSimple_OriginalPhase3_ThreeModels_FromScratch.ipynb` (Phi-3 selected, exact original Phase-3 fitting logic; pinned model revision and paper-only toggles).
- `ULM_Phi3_IFEval_CCT_source_Canonical_server Jupyter.ipynb` (historical Phase-1 dependency removed; published 541-source data, conservative rewrites, same final paper readouts and checks).
- `ULM_Phi3_FollowBench_Controlled_server Jupyter_Fixed.ipynb` (latest fixed FollowBench-derived controlled-constraint implementation, full grouping and control logic).

The final server Jupyter notebook is constructed from those original source cells. The only intentional protocol-level changes are disabling optional exploratory sweeps / native full-benchmark generation and using the already-frozen Phi-3 commit rather than resolving a mutable latest revision. **The notebook's Python syntax and data-flow contracts have been checked locally, but no A100/H100 inference has been executed in the creation environment.** Do not claim bitwise reproduction or successful end-to-end execution until the GPU run and independent checker audits finish.

# Part III. Counterfactual Compliance Transfer (CCT): task-family-out-of-fold 2 × 2 diagnostic

This section needs **only the user-supplied CCT data ZIP**, already verified in Cell 0. It constructs all 534 original task/constraint combinations and all four conditions (O/R1/C0/C1), with **282 independent source-task families**. All responses and pre-generation states are generated on this server with the pinned Phi-3 checkpoint and official checker, and cached atomically for resume.

**The critical design difference from the archived CCT run:** because the historic frozen source probes and private training-state archives cannot be downloaded, this section fits fresh **B0, B2, MT0, MT2** probes using only *O/R1 training families*, grouped into five outer folds. C0/C1 outcomes are never used in fitting or selecting readouts. These probes use the same convex, weighted-BCE regularized readout *family* and hyperparameter grids as the main source studies. The CCT outcome tables are thus an independently reproducible **new protocol**, not a claim of exact agreement with the old CCT numerical results.

The main paper's claims C1 and C2 are supported by Parts I–IV. This experiment tests the stronger C3 question by contrasting changes to wording and to the required behavior. The notebook includes task-only lexical control, paired/clustered uncertainty, response checker verification, Counterfactual *prompt* differences are not neural causal interventions.

In [ ]:
if not REPLAY_COMPLETE:
    from ulm.models import get_spec
    SPEC=get_spec('phi3')
    from ulm.cct import render_common
    CCT_MODULE_DIR=ULM_WORKDIR/'cct_modules'
    CCT_MODULE_DIR.mkdir(parents=True,exist_ok=True)
    (CCT_MODULE_DIR/'cct_common.py').write_text(render_common(SPEC,MODEL_REV))


In [ ]:
if not REPLAY_COMPLETE:
    from ulm.cct import render_inference
    CCT_MODULE_DIR=ULM_WORKDIR/'cct_modules'
    CCT_MODULE_DIR.mkdir(parents=True,exist_ok=True)
    (CCT_MODULE_DIR/'cct_inference.py').write_text(render_inference(SPEC))


In [ ]:
if not REPLAY_COMPLETE:
    import sys, os, json, hashlib, zipfile, itertools, gc
    from pathlib import Path
    from collections import defaultdict
    import pandas as pd, numpy as np, torch
    from tqdm.auto import tqdm
    from scipy.special import expit
    from sklearn.model_selection import KFold, train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, brier_score_loss
    from IPython.display import display
    if str(CCT_MODULE_DIR) not in sys.path:sys.path.insert(0,str(CCT_MODULE_DIR))
    from cct_common import load_cct,checker_module,hash_prompt,savejson,REVISION
    from cct_inference import CCTInference
    assert MODEL_REV==REVISION, 'CCT source model revision does not match main experiments'
    cct_new,cct_family,cct_protocol=load_cct(CCT_ZIP)
    CCT_ROOT=ULM_WORKDIR/'CCT_independent_refit'
    CCT_ROOT.mkdir(parents=True,exist_ok=True)
    CCT_CHECK=checker_module(CCT_ROOT)
    
    source_family_for_case={(r['section'],int(r['task_id'])):str(r['task_family_id']) for r in cct_new}
    assert len(set(source_family_for_case.values()))==282, 'Expected 282 independent CCT base-task families'
    assert all(sum(1 for sec,task in source_family_for_case if (sec,task)==key)==1 for key in cct_family)
    CCT_CASES=[]
    for (sec,task),f in sorted(cct_family.items()):
        family=source_family_for_case[(sec,task)]
        typ=('startend:end_checker' if sec in 'AC' else 'detectable_content:number_placeholders')
        new_example=next(r for r in cct_new if r['section']==sec and int(r['task_id'])==task)
        for cond in ('O','R1','C0','C1'):
            p=f[f'{cond}_prompt'];kw=f['original_kwargs'] if cond in ('O','R1') else f['counterfactual_kwargs']
            CCT_CASES.append({'id':f'{sec}_{task}_{cond}', 'section':sec,'task_id':int(task),
                'condition':cond,'task_family_id':family,'type':typ,'instruction_type':typ,
                'prompt':p,'prompt_sha256':hash_prompt(p),'target_kwargs':kw,
                'original_kwargs':f['original_kwargs'],'base_task':f['base_task'],
                'generation_budget':int(new_example['generation_budget']),
                'max_input_tokens':int(new_example.get('max_input_tokens',1024)),
                'fold':int(f['cct_fold'])})
    assert len(CCT_CASES)==2136 and len({x['id'] for x in CCT_CASES})==2136
    assert all(x['generation_budget']==(1024 if x['section'] in 'AB' else 1536) for x in CCT_CASES)
    cct_manifest={'source_zip_sha256':hashlib.sha256(CCT_ZIP.read_bytes()).hexdigest(),
     'checkpoint_revision':MODEL_REV,'independent_training':'O canonical, R1 augmentation from other families only',
     'heldout_requirement_conditions':['C0','C1'],'folds':5,
     'augmentation_training_source':'CCT O/R1 on training families, not archived source study probes',
     'method_names':['B0','B2','MT0','MT2'],'probe_layer':14,'generation_policy':'BF16 SDPA greedy, source per-case caps'}
    from cct_common import savejson
    manifest_path=CCT_ROOT/'frozen_protocol.json'
    if manifest_path.exists():assert json.loads(manifest_path.read_text())==cct_manifest, 'CCT run provenance changed'
    else:savejson(manifest_path,cct_manifest)
    print('CCT verified:',len(CCT_CASES),'conditions,',len(set(source_family_for_case.values())),'source families')


### 3.1 Cache O/R1 labels and states; fit probes before generating C0/C1

Each family is held out from its own training probe. B0/MT0 train on canonical prompts, B2/MT2 on canonical plus R1 formulations in **training families only**. Only after training and saving the models will the notebook generate C0/C1 responses. This sequence avoids inadvertent test-set hyperparameter selection.

In [ ]:
if not REPLAY_COMPLETE:
    cct_runner=CCTInference(CCT_ROOT,CCT_CHECK)
    CCT_TYPES=('startend:end_checker','detectable_content:number_placeholders')
    CCT_TYPE_ID={t:i for i,t in enumerate(CCT_TYPES)}
    CCT_SEED=20261017
    CCT_FOLDS=5
    CCT_METHODS=('B0','B2','MT0','MT2')
    CCT_GROUPS=np.array(sorted(set(source_family_for_case.values())))
    cct_fold={item['task_family_id']:int(item['fold']) for item in CCT_CASES}
    assert len(cct_fold)==282
    assert set(cct_fold.values())==set(range(CCT_FOLDS))
    cct_split=[]
    for fold in range(CCT_FOLDS):
        test=np.flatnonzero([cct_fold[str(group)]==fold for group in CCT_GROUPS])
        train=np.flatnonzero([cct_fold[str(group)]!=fold for group in CCT_GROUPS])
        cct_split.append((train,test))
    cct_labels=[];cct_states=[]
    for item in tqdm([r for r in CCT_CASES if r['condition'] in ('O','R1')],desc='CCT source O/R1 (new responses + states)'):
        gen,_=cct_runner.get_response(item);st,_=cct_runner.get_state(item)
        cct_labels.append({k:item[k] for k in ('id','section','task_id','condition','task_family_id','type','base_task','prompt')}
             |{'label':int(gen['label']),'capped':int(gen['capped']),'tokens':int(gen['tokens']),
               'fold':cct_fold[item['task_family_id']]})
        cct_states.append(np.asarray(st[0],np.float32))
    CCT_TR=pd.DataFrame(cct_labels)
    CCT_X=np.stack(cct_states).astype(np.float32)
    assert len(CCT_TR)==1068 and len(CCT_X)==1068
    CCT_TR.to_csv(CCT_ROOT/'source_train_labels.csv',index=False)
    np.save(CCT_ROOT/'source_train_states_layer14.npy',CCT_X.astype(np.float16))
    
    CCT_SOURCE_SHA=hashlib.sha256(CCT_TR[['id','label','fold']].to_csv(index=False).encode()+CCT_X.tobytes()).hexdigest()
    (CCT_ROOT/'source_training_fingerprint.sha256').write_text(CCT_SOURCE_SHA+'\n')
    CCT_TR['tid']=CCT_TR.type.map(CCT_TYPE_ID).astype(int)
    assert not CCT_TR.duplicated(['section','task_id','condition']).any()
    print('CCT training-source labels measured:',len(CCT_TR),'| success',round(float(CCT_TR.label.mean()),3))

In [ ]:
if not REPLAY_COMPLETE:
    
    class CCTProbe:
        def __init__(self,mean,scale,w,delta,offset,b):
            self.mean=np.asarray(mean,np.float32);self.scale=np.asarray(scale,np.float32)
            self.w=np.asarray(w,np.float32);self.delta=np.asarray(delta,np.float32)
            self.offset=np.asarray(offset,np.float32);self.b=float(b)
        def predict(self,X,T):
            X=np.asarray(X,np.float32).reshape(-1,3072);T=np.asarray(T,int).reshape(-1)
            assert len(X)==len(T) and len(self.delta)==2
            z=(X-self.mean)/self.scale
            return expit(np.clip(np.sum(z*(self.w+self.delta[T]),axis=1)+self.b+self.offset[T],-35,35))
        def save(self,path):
            with open(path,'wb') as out:np.savez_compressed(out,mean=self.mean,scale=self.scale,w=self.w,
                delta=self.delta,offset=self.offset,b=np.array(self.b))
        @classmethod
        def load(cls,path):
            with np.load(path,allow_pickle=False) as x:return cls(x['mean'],x['scale'],x['w'],x['delta'],x['offset'],float(x['b']))

    def fit_cct(training_groups,method,C,penalty=16,max_iter=80):
        allowed=CCT_TR.task_family_id.isin(training_groups)
        chosen=allowed & (CCT_TR.condition.eq('O') if method in ('B0','MT0') else True)
        idx=np.flatnonzero(chosen.to_numpy())
        orig=np.flatnonzero((allowed&CCT_TR.condition.eq('O')).to_numpy())
        norm=StandardScaler().fit(CCT_X[orig]);scale=np.maximum(norm.scale_,1e-6).astype(np.float32)
        XX=((CCT_X[idx]-norm.mean_)/scale).astype(np.float32)
        yy=CCT_TR.iloc[idx].label.to_numpy(np.float32)
        typ=CCT_TR.iloc[idx].tid.to_numpy(np.int64)
        assert len(np.unique(yy))==2,('Missing classes for CCT training subset',method)
        groups=CCT_TR.iloc[idx].groupby(['task_family_id','type']).size().to_dict()
        weights=np.array([1/groups[(r.task_family_id,r.type)] for r in CCT_TR.iloc[idx].itertuples()],np.float32)
        weights/=weights.sum()
        device=torch.device('cuda:0')
        xt=torch.from_numpy(XX).to(device); yt=torch.from_numpy(yy).to(device)
        tt=torch.from_numpy(typ).to(device);sw=torch.from_numpy(weights).to(device)
        w=torch.nn.Parameter(torch.zeros(3072,device=device));b=torch.nn.Parameter(torch.zeros((),device=device))
        typed=method.startswith('MT')
        d=torch.nn.Parameter(torch.zeros((2,3072),device=device)) if typed else None
        o=torch.nn.Parameter(torch.zeros(2,device=device)) if typed else None
        opt=torch.optim.LBFGS([w,b]+([d,o] if typed else []),lr=1,max_iter=max_iter,
                               line_search_fn='strong_wolfe',tolerance_grad=2e-5,tolerance_change=1e-9)
        ridge=1/(2*float(C)*len(groups))
        def closure():
            opt.zero_grad()
            logits=(xt*(w+(d[tt] if typed else 0))).sum(1)+b
            if typed:logits=logits+o[tt]
            loss=(sw*torch.nn.functional.binary_cross_entropy_with_logits(logits,yt,reduction='none')).sum()+ridge*w.square().sum()
            if typed:loss=loss+ridge*float(penalty)*d.square().sum()+ridge*o.square().sum()
            assert torch.isfinite(loss).item()
            loss.backward();return loss
        opt.step(closure)
        output=CCTProbe(norm.mean_,scale,w.detach().cpu().numpy(),
          d.detach().cpu().numpy() if typed else np.zeros((2,3072),np.float32),
          o.detach().cpu().numpy() if typed else np.zeros(2,np.float32),float(b.detach().cpu()))
        del xt,yt,tt,sw,w,b,d,o,opt
        return output

    def cct_macro(probe,groups,conditions=('O','R1')):
        subset=CCT_TR[CCT_TR.task_family_id.isin(groups)&CCT_TR.condition.isin(conditions)]
        scores=[]
        for typ in CCT_TYPES:
            for cond in conditions:
                rows=subset[(subset.type==typ)&(subset.condition==cond)]
                if rows.label.nunique()<2:continue
                inds=rows.index.to_numpy(int)
                values=probe.predict(CCT_X[inds],np.full(len(inds),CCT_TYPE_ID[typ]))
                scores.append(roc_auc_score(rows.label,values))
        return float(np.mean(scores)) if scores else float('nan')

    CCT_PROBE_DIR=CCT_ROOT/'fold_probes';CCT_PROBE_DIR.mkdir(parents=True,exist_ok=True)
    CCT_FITTED={};CCT_TUNING=[]
    for fold,(tri,tei) in enumerate(cct_split):
        trainval=CCT_GROUPS[tri];held=CCT_GROUPS[tei]
        inner_tr,inner_val=train_test_split(trainval,test_size=.25,random_state=CCT_SEED+fold)
        base_orig=None
        for method in CCT_METHODS:
            dest=CCT_PROBE_DIR/f'{fold}_{method}.npz'
            selection=CCT_PROBE_DIR/f'{fold}_{method}.json'
            if dest.exists() and selection.exists():
                info=json.loads(selection.read_text())
                assert info['cct_protocol_sha256']==hashlib.sha256(json.dumps(cct_manifest,sort_keys=True).encode()).hexdigest()
                assert info['cct_source_label_state_sha256']==CCT_SOURCE_SHA, 'CCT refit weights do not match current generated source states/labels'
                probe=CCTProbe.load(dest);CCT_FITTED[(fold,method)]=probe;CCT_TUNING.append(info)
                if method=='B0':base_orig=info['original_val_AUROC']
                continue
            candidates=[]
            Cs=(0.03,0.1,0.3);penalties=(4.,16.)
            grid=[(c,16.) for c in Cs] if method in ('B0','B2') else list(itertools.product(Cs,penalties))
            for C,penalty in grid:
                p=fit_cct(inner_tr,method,C,penalty,max_iter=80)
                transfer=cct_macro(p,inner_val,('R1',))
                original=cct_macro(p,inner_val,('O',))
                if np.isfinite(transfer) and np.isfinite(original):
                    candidates.append((transfer,original,float(C),float(penalty)))
            assert candidates,(fold,method,'No class-balanced inner folds: CCT readout invalid')
            eligible=[e for e in candidates if base_orig is None or e[1]>=base_orig-.02]
            
            candidates_to_consider=eligible if eligible else candidates
            best=max(candidates_to_consider,key=lambda x:(x[0],x[1],-x[2]))
            if method=='B0':base_orig=best[1]
            p=fit_cct(trainval,method,best[2],best[3],max_iter=110)
            p.save(dest)
            setting={'fold':fold,'method':method,'C':best[2],'penalty':best[3],
              'original_val_AUROC':best[1],'rephrase_val_AUROC':best[0],
              'n_train_families':len(trainval),'n_test_families':len(held),
              'cct_protocol_sha256':hashlib.sha256(json.dumps(cct_manifest,sort_keys=True).encode()).hexdigest(),
              'cct_source_label_state_sha256':CCT_SOURCE_SHA}
            savejson(selection,setting)
            CCT_FITTED[(fold,method)]=p;CCT_TUNING.append(setting)
        print('Frozen CCT probe fold',fold,'| readouts',len(CCT_METHODS))
    assert len(CCT_FITTED)==20
    pd.DataFrame(CCT_TUNING).to_csv(CCT_ROOT/'probe_tuning_from_O_R1_only.csv',index=False)
    print('CCT readouts frozen before any counterfactual C0/C1 response generation')

### 3.2 Generate C0/C1, verify every response, and score all four conditions out of fold

Generation and activation extraction are separately cached, with strict prompt/rule/revision fingerprints. Every CCT condition is evaluated with the same fold's readout. An explicit task-only text baseline is trained only on O examples from different families.

In [ ]:
if not REPLAY_COMPLETE:
    
    for item in tqdm([r for r in CCT_CASES if r['condition'] in ('C0','C1')],desc='CCT withheld C0/C1 generation'):
        gen,_=cct_runner.get_response(item);_,_=cct_runner.get_state(item)
        assert int(gen['label'])==CCT_CHECK(item['instruction_type'],item['target_kwargs'],gen['response'])
    CCT_PROBE_PRED=[]
    for item in tqdm(CCT_CASES,desc='CCT all-condition out-of-fold scores'):
        gen,_=cct_runner.get_response(item);st,_=cct_runner.get_state(item)
        typ=CCT_TYPE_ID[item['instruction_type']];fold=cct_fold[item['task_family_id']]
        record={k:item[k] for k in ('id','section','task_id','condition','task_family_id','type','base_task','prompt')}
        record.update({'fold':fold,'label':int(gen['label']),'capped':int(gen['capped']),
          'tokens':int(gen['tokens']),'input_tokens':int(gen['input_tokens']),
          'original_grade_same_response':int(gen['original_grade_same_response'])})
        for method in CCT_METHODS:
            record[method]=float(CCT_FITTED[(fold,method)].predict(st[0],np.array([typ]))[0])
        CCT_PROBE_PRED.append(record)
    CCT_PRED=pd.DataFrame(CCT_PROBE_PRED)
    assert len(CCT_PRED)==2136
    assert CCT_PRED.groupby(['section','condition']).size().to_dict()=={
     (sec,cond):n for sec,n in {'A':102,'B':102,'C':161,'D':169}.items()
     for cond in ('O','R1','C0','C1')}
    assert not CCT_PRED.duplicated(['section','task_id','condition']).any()
    assert CCT_PRED.groupby('task_family_id').fold.nunique().max()==1
    CCT_PRED.to_csv(CCT_ROOT/'all_four_condition_oof_predictions.csv',index=False)
    
    task_scores=[]
    for fold in range(5):
        tr=CCT_PRED[(CCT_PRED.fold!=fold)&CCT_PRED.condition.eq('O')]
        te=CCT_PRED[CCT_PRED.fold==fold]
        assert tr.label.nunique()==2
        vector=TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,max_features=25000)
        Xtr=vector.fit_transform(tr.base_task.astype(str))
        learner=LogisticRegression(max_iter=400,solver='liblinear',C=1.0)
        learner.fit(Xtr,tr.label)
        scores=learner.predict_proba(vector.transform(te.base_task.astype(str)))[:,1]
        task_scores.extend(zip(te.id.tolist(),scores.tolist()))
    prior=pd.DataFrame(task_scores,columns=['id','task_only'])
    CCT_PRED=CCT_PRED.merge(prior,on='id',validate='one_to_one')
    assert CCT_PRED.task_only.notna().all()
    CCT_PRED.to_csv(CCT_ROOT/'all_four_condition_oof_predictions.csv',index=False)
    
    print('CCT successful: all conditions have verified labels, frozen-fold model scores, and task-only controls.')

### 3.3 CCT results needed for the final paper: AUROC, paired sign alignment, and uncertainty

Primary descriptive quantities are held-out C1 discrimination and the within-family changes in predicted success versus the observed 0→1 or 1→0 compliance change. All inference labels are produced by deterministic verification. **This is a predictive test, not a causal mechanism study.** Sections are reported individually because their class balance and task compatibility differ. The combined four-section result is secondary.

In [ ]:
if not REPLAY_COMPLETE:
    CCT_STATS=[]
    for (sec,cond),g in CCT_PRED.groupby(['section','condition']):
        for m in (*CCT_METHODS,'task_only'):
            y=g.label.to_numpy(int)
            CCT_STATS.append({'section':sec,'condition':cond,'method':m,'n':len(g),
              'positive':int(y.sum()),'negative':int(len(y)-y.sum()),
              'AUROC':roc_auc_score(y,g[m]) if np.unique(y).size==2 else np.nan,
              'Brier':brier_score_loss(y,g[m])})
    CCT_STATS=pd.DataFrame(CCT_STATS)
    CCT_STATS.to_csv(CCT_ROOT/'four_condition_AUROC_by_section.csv',index=False)
    CCT_WIDE=CCT_PRED.pivot(index=['section','task_id','task_family_id','type','fold'],columns='condition',
       values=['label','B0','B2','MT0','MT2','task_only'])
    assert CCT_WIDE.notna().all().all()
    CCT_PAIRS=[]
    for (sec,task,family,typ,fold),r in CCT_WIDE.iterrows():
        entry={'section':sec,'task_id':task,'family':family,'type':typ,'fold':int(fold)}
        for a,b,name in [('O','R1','wording_original_rule'),('C0','C1','wording_changed_rule'),
                         ('O','C0','requirement_original_wording'),('R1','C1','requirement_reworded')]:
            dy=int(r[('label',b)]-r[('label',a)])
            entry[f'{name}_y_diff']=dy
            for method in (*CCT_METHODS,'task_only'):
                ds=float(r[(method,b)]-r[(method,a)])
                entry[f'{name}_{method}_score_diff']=ds
                entry[f'{name}_{method}_correct_on_flip']=int(ds*dy>0) if dy else np.nan
        entry['factorial_y_interaction']=int(r[('label','C1')]-r[('label','C0')]
                                             -r[('label','R1')]+r[('label','O')])
        CCT_PAIRS.append(entry)
    CCT_PAIRS=pd.DataFrame(CCT_PAIRS)
    CCT_PAIRS.to_csv(CCT_ROOT/'paired_2x2_changes.csv',index=False)
    
    rng=np.random.default_rng(20261017)
    FAMS=np.array(sorted(CCT_PAIRS.family.unique()));FIDX={x:i for i,x in enumerate(FAMS)}
    W=rng.multinomial(len(FAMS),np.full(len(FAMS),1/len(FAMS)),size=1000)
    CCT_FLIP=[]
    for section in ['A','B','C','D','all']:
        df=CCT_PAIRS if section=='all' else CCT_PAIRS[CCT_PAIRS.section==section]
        for name in ['wording_original_rule','wording_changed_rule','requirement_original_wording','requirement_reworded']:
            dy=df[f'{name}_y_diff'].to_numpy(int);changed=(dy!=0)
            family_index=np.array([FIDX[x] for x in df.family]);family_flips=np.bincount(family_index,weights=changed.astype(float),minlength=len(FAMS))
            up=int((dy==1).sum());down=int((dy==-1).sum())
            for method in (*CCT_METHODS,'task_only'):
                ds=df[f'{name}_{method}_score_diff'].to_numpy(float)
                right=(ds*dy>0)&changed
                family_right=np.bincount(family_index,weights=right.astype(float),minlength=len(FAMS))
                a=W@family_right;b=W@family_flips
                bootstrap=np.divide(a,b,out=np.full(len(W),np.nan),where=b>0)
                good=bootstrap[np.isfinite(bootstrap)]
                CCT_FLIP.append({'section':section,'contrast':name,'method':method,
                  'n_pairs':len(df),'n_flips':int(changed.sum()),'failure_to_success':up,'success_to_failure':down,
                  'flip_direction_acc':float(right.sum()/changed.sum()) if changed.sum() else np.nan,
                  'ci_low':float(np.quantile(good,.025)) if len(good) else np.nan,
                  'ci_high':float(np.quantile(good,.975)) if len(good) else np.nan,
                  'majority_direction_descriptive':max(up,down)/int(changed.sum()) if changed.sum() else np.nan})
    CCT_FLIP=pd.DataFrame(CCT_FLIP)
    CCT_FLIP.to_csv(CCT_ROOT/'pairwise_flip_direction_clustered_CI.csv',index=False)
    print('CCT C1 AUC:')
    display(CCT_STATS[(CCT_STATS.condition=='C1')][['section','method','n','positive','negative','AUROC']].round(3))
    print('CCT wording-change flip metrics (MT2 and task-only controls):')
    display(CCT_FLIP[(CCT_FLIP.section=='all')&(CCT_FLIP.method.isin(['MT2','task_only']))].round(3))


### 3.4 Final server manifest and portable output archive

All source-study and CCT results, frozen protocol metadata, paired outcomes, and method-level diagnostic files stay in `ULM_WORKDIR`. The final package includes CSVs, JSONs, analysis code in the notebook, **not model weights or raw HF checkpoint shards**. Keep the working directory to resume interrupted runs.

In [ ]:
if not REPLAY_COMPLETE:
    import zipfile,hashlib,os,json
    manifest={
     'checkpoint':'microsoft/Phi-3-mini-128k-instruct','revision':MODEL_REV,
     'benchmark_zip':str(CCT_ZIP),'benchmark_sha256':hashlib.sha256(CCT_ZIP.read_bytes()).hexdigest(),
     'source_results':{'IFEval-Simple':str(SIMPLE_RESULTS),
                       'FollowBench-derived':str(FOLLOWBENCH_RESULTS)},
     'cct_refit_protocol':'independent CCT five-fold O/R1-trained refit; NOT byte-identical archived source-study probes',
     'claims':['formulation_transfer','type-aware_over_equally_augmented_shared','paired_individual_change_prediction'],
     'model_weights_in_archive':False,
     'input_count':len(CCT_CASES),'source_families':len(FAMS)}
    (ULM_WORKDIR/'RUN_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
    package=ULM_WORKDIR/'Phi3_Main_and_CCT_Final_Tables.zip'
    with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED) as z:
        include=[ULM_WORKDIR/'RUN_MANIFEST.json']
        include+=list((ULM_WORKDIR/'final_tables').glob('*.csv'))
        include+=list((ULM_WORKDIR/'final_tables').glob('*.json'))
        include+=list(CCT_ROOT.glob('*.csv'))
        include+=list(CCT_ROOT.glob('frozen_protocol.json'))
        include+=list(CCT_ROOT.glob('probe_tuning_from_O_R1_only.csv'))
        for p in include:
            if p.is_file():z.write(p,arcname=str(p.relative_to(ULM_WORKDIR)))
    print('Completed.',package)
    print('Resume caches, full response texts, hidden states and probes stay under',ULM_WORKDIR)
    print('No previous Drive archives, manual downloads, Colab mount, or interactive upload was required.')

## Completion and fast replay

This cell writes a verified complete-run marker only after all three settings and CCT have finished and the required output tables exist. Subsequent **Run All** checks the saved outputs before installing packages or loading a model. Partial runs resume using individual source caches.

In [ ]:
if not REPLAY_COMPLETE:
    
    
    import hashlib,json,os
    from pathlib import Path
    _all_in_root=Path(ULM_WORKDIR)
    _completed_required=[
     'final_tables/paper_table_main_AUROC.csv',
     'final_tables/paper_table_main_comparisons.csv',
     'final_tables/all_out_of_fold_predictions.csv',
     'CCT_independent_refit/all_four_condition_oof_predictions.csv',
     'CCT_independent_refit/paired_2x2_changes.csv',
     'CCT_independent_refit/pairwise_flip_direction_clustered_CI.csv',
     'CCT_independent_refit/frozen_protocol.json',
     'CCT_independent_refit/probe_tuning_from_O_R1_only.csv',
     'RUN_MANIFEST.json'
    ]
    for _r in _completed_required:
        assert (_all_in_root/_r).is_file() and (_all_in_root/_r).stat().st_size>0, 'Cannot mark complete: '+_r
    _complete_files={}
    for _p in sorted(_all_in_root.rglob('*')):
        if not _p.is_file() or _p.name in ('COMPLETED_RUN.json','COMPLETED_RUN.json.tmp'):continue
        _rel=str(_p.relative_to(_all_in_root))
        _stat=_p.stat()
        _entry={'size':_stat.st_size}
        
        
        
        if (_rel.startswith('final_tables/') or _rel.startswith('CCT_independent_refit/')
            and not ('/response_cache/' in _rel or '/activation_cache/' in _rel or '/fold_probes/' in _rel)
            or _p.suffix=='.json' and '/cache/' not in _rel
            or _p.suffix=='.npz' and ('/probes/' in _rel or '/fold_probes/' in _rel)):
            _entry['sha256']=_fingerprint_file(_p)
        _complete_files[_rel]=_entry
    assert len(_complete_files)>40, 'Completed run is missing most of its saved intermediate files'
    _complete={'version':'verified-v2','model':_MODEL_LABEL,'notebook_sha256':_CODE_RELEASE_SHA,
     'cct_sha256':_fingerprint_file(_ULM_DATA),'files':_complete_files,
     'n_files':len(_complete_files),'completion_requires':_completed_required}
    _tmp=_RUN_MARKER.with_suffix('.json.tmp')
    _tmp.write_text(json.dumps(_complete,indent=2,sort_keys=True),encoding='utf-8')
    os.replace(_tmp,_RUN_MARKER)
    print('VERIFIED RUN COMPLETED:',len(_complete_files),'saved files; subsequent Run All uses fast replay.')
